# MovieLens-100K Novelty Experiments (PopDebias + ColdBridge)

This notebook runs the novelty pipeline described in `kaggle/novelty` on MovieLens-100K using GPU-friendly BGE-M3 embeddings.

It will generate:
- `results/movielens_100k/full_results.csv`
- `results/movielens_100k/best_model.txt`
- `results/figures/*.png`

In [1]:
import importlib.util
import os
import subprocess
import sys

def pip_install(*packages):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)

def ensure_package(import_name: str, pip_name: str | None = None):
    if importlib.util.find_spec(import_name) is None:
        pip_install(pip_name or import_name)

run_mode_env = os.environ.get("NOVELTY_RUN_MODE", "v3_fast").strip().lower()

# Only install what is missing to reduce Kaggle startup time.
ensure_package("numpy")
ensure_package("pandas")
ensure_package("matplotlib")
ensure_package("sklearn", "scikit-learn")
ensure_package("tensorflow", "tensorflow==2.19.0")
ensure_package("torch")
ensure_package("transformers")
ensure_package("sentencepiece")

if run_mode_env == "full":
    ensure_package("sentence_transformers", "sentence-transformers")
else:
    print(f"Skipping sentence-transformers install for NOVELTY_RUN_MODE={run_mode_env}.")

print("Dependency check complete.")

ERROR: Operation cancelled by user


KeyboardInterrupt: 

In [ ]:
import math
import random
import time
import urllib.request
import zipfile
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.decomposition import PCA
from tensorflow import keras
from transformers import AutoModel, AutoTokenizer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception:
        pass

WORK_DIR = Path("/kaggle/working/llmseqrec_ml100k_single_notebook")
DATA_DIR = WORK_DIR / "data"
EMB_DIR = WORK_DIR / "embeddings" / "movielens_100k"
RESULTS_DIR = WORK_DIR / "results" / "movielens_100k"
FIG_DIR = WORK_DIR / "results" / "figures"

for p in [DATA_DIR, EMB_DIR, RESULTS_DIR, FIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

DATASET_URL = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
SESSION_GAP_MINUTES = 30
MIN_SESSION_LEN = 2
VAL_FRAC = 0.1
TEST_FRAC = 0.2
SEQ_LEN = 20

NUM_EPOCHS = 4
BATCH_SIZE = 256
LEARNING_RATE = 1e-3
EMB_DIM = 64

TOP_K = 20
CANDIDATE_K = 300
ALPHAS = [0.1, 0.3, 0.5, 0.7]
TAUS = [2, 3, 5, 10, 15]
DECAYS = [0.5, 0.7, 0.8, 0.9, 1.0]
LONG_TAIL_THRESHOLD = 500

print("Using work dir:", WORK_DIR)
print("TensorFlow version:", tf.__version__)
print("GPU devices:", gpus)

2026-04-20 13:00:33.199061: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776690033.224941      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776690033.233433      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776690033.252463      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776690033.252492      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776690033.252495      55 computation_placer.cc:177] computation placer alr

Using work dir: /kaggle/working/llmseqrec_ml100k_single_notebook
TensorFlow version: 2.19.0
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
def download_ml100k(data_dir: Path, dataset_url: str) -> tuple[Path, Path]:
    udata_path = data_dir / "ml-100k" / "u.data"
    uitem_path = data_dir / "ml-100k" / "u.item"
    if udata_path.exists() and uitem_path.exists():
        return udata_path, uitem_path

    data_dir.mkdir(parents=True, exist_ok=True)
    archive_path = data_dir / "ml-100k.zip"
    if not archive_path.exists():
        urllib.request.urlretrieve(dataset_url, archive_path)

    with zipfile.ZipFile(archive_path, "r") as zf:
        zf.extractall(data_dir)

    if not (udata_path.exists() and uitem_path.exists()):
        raise FileNotFoundError("MovieLens-100K files not found after extraction.")
    return udata_path, uitem_path


def load_interactions(udata_path: Path) -> pd.DataFrame:
    return pd.read_csv(
        udata_path,
        sep="\t",
        names=["UserId", "ItemId", "Rating", "Timestamp"],
        engine="python",
    )


def sessionize(interactions: pd.DataFrame, gap_minutes: int, min_session_len: int) -> pd.DataFrame:
    df = interactions.copy()
    df["Timestamp"] = df["Timestamp"].astype(int)
    df["ItemId"] = df["ItemId"].astype(int)
    df = df.sort_values(["UserId", "Timestamp", "ItemId"]).reset_index(drop=True)

    gap_seconds = gap_minutes * 60
    user_changed = df["UserId"].ne(df["UserId"].shift(1))
    ts_gap = df["Timestamp"].diff().fillna(0)
    new_session = user_changed | (ts_gap > gap_seconds)
    df["SessionId"] = new_session.cumsum().astype(int)

    valid = df.groupby("SessionId").size()
    valid_ids = valid[valid >= min_session_len].index
    df = df[df["SessionId"].isin(valid_ids)].copy()

    remap = {old: i + 1 for i, old in enumerate(sorted(df["SessionId"].unique()))}
    df["SessionId"] = df["SessionId"].map(remap).astype(int)

    df["Time"] = pd.to_datetime(df["Timestamp"], unit="s", utc=True).dt.tz_localize(None)
    df["Time"] = df["Time"].dt.strftime("%Y-%m-%d %H:%M:%S.%f")
    df["Reward"] = df["Rating"].astype(float)
    return df[["SessionId", "ItemId", "Time", "Reward"]].sort_values(["SessionId", "Time", "ItemId"]).reset_index(drop=True)


def temporal_split(sessions_df: pd.DataFrame, val_frac: float, test_frac: float) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    df = sessions_df.copy()
    df["Time"] = pd.to_datetime(df["Time"])

    session_end = df.groupby("SessionId")["Time"].max().sort_values()
    session_ids = session_end.index.to_numpy()
    n = len(session_ids)

    n_test = max(1, int(round(n * test_frac)))
    n_val = max(1, int(round(n * val_frac)))
    n_train = n - n_val - n_test
    if n_train <= 0:
        raise ValueError("Split produced empty train partition.")

    train_ids = set(session_ids[:n_train])
    val_ids = set(session_ids[n_train:n_train + n_val])
    test_ids = set(session_ids[n_train + n_val:])

    train_df = df[df["SessionId"].isin(train_ids)].copy()
    val_df = df[df["SessionId"].isin(val_ids)].copy()
    test_df = df[df["SessionId"].isin(test_ids)].copy()

    return (
        train_df.sort_values(["SessionId", "Time", "ItemId"]).reset_index(drop=True),
        val_df.sort_values(["SessionId", "Time", "ItemId"]).reset_index(drop=True),
        test_df.sort_values(["SessionId", "Time", "ItemId"]).reset_index(drop=True),
    )


def build_cases(split_df: pd.DataFrame, n_withheld: int = 1) -> tuple[dict[int, np.ndarray], dict[int, np.ndarray], dict[int, int]]:
    prompts: dict[int, np.ndarray] = {}
    gts: dict[int, np.ndarray] = {}
    lengths: dict[int, int] = {}

    for sid, cur in split_df.groupby("SessionId"):
        items = cur["ItemId"].to_numpy(dtype=int)
        if len(items) <= n_withheld:
            continue
        prompt = items[:-n_withheld]
        gt = items[-n_withheld:]
        if len(prompt) == 0:
            continue
        sid = int(sid)
        prompts[sid] = prompt
        gts[sid] = gt
        lengths[sid] = len(prompt)
    return prompts, gts, lengths


def load_item_metadata(uitem_path: Path) -> pd.DataFrame:
    genre_cols = [
        "unknown", "action", "adventure", "animation", "childrens", "comedy",
        "crime", "documentary", "drama", "fantasy", "film_noir", "horror",
        "musical", "mystery", "romance", "sci_fi", "thriller", "war", "western",
    ]
    cols = ["ItemId", "title", "release_date", "video_release_date", "imdb_url", *genre_cols]
    df = pd.read_csv(
        uitem_path,
        sep="|",
        names=cols,
        encoding="latin-1",
        usecols=list(range(len(cols))),
        engine="python",
    )

    def build_text(row: pd.Series) -> str:
        genres = [g.replace("_", " ") for g in genre_cols if int(row[g]) == 1]
        genre_text = ", ".join(genres) if genres else "unknown"
        return f"{row['title']}. Genres: {genre_text}."

    df["item_text"] = df.apply(build_text, axis=1)
    return df[["ItemId", "title", "item_text"]]


def normalize_rows(x: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(x, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return x / norms


def encode_bge_cpu(item_ids: np.ndarray, item_text_df: pd.DataFrame, output_npy: Path, batch_size: int = 32, max_length: int = 256) -> np.ndarray:
    output_npy.parent.mkdir(parents=True, exist_ok=True)

    text_lookup = item_text_df.set_index("ItemId")["item_text"].to_dict()
    texts = [text_lookup.get(int(i), f"Movie item {int(i)}.") for i in item_ids]

    if output_npy.exists():
        cached = np.load(output_npy)
        if cached.shape[0] == len(item_ids):
            return cached

    import torch
    import torch.nn.functional as F

    device = "cpu"
    tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3")

    try:
        model = AutoModel.from_pretrained("BAAI/bge-m3", dtype=torch.float32)
    except TypeError:
        model = AutoModel.from_pretrained("BAAI/bge-m3", torch_dtype=torch.float32)
    model.to(device)
    model.eval()

    all_chunks = []
    for start in range(0, len(texts), batch_size):
        cur = texts[start:start + batch_size]
        encoded = tokenizer(
            cur,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}

        with torch.no_grad():
            out = model(**encoded)
            token_emb = out.last_hidden_state
            mask = encoded["attention_mask"].unsqueeze(-1).expand(token_emb.size()).float()
            pooled = (token_emb * mask).sum(dim=1) / torch.clamp(mask.sum(dim=1), min=1e-9)
            pooled = F.normalize(pooled, p=2, dim=1)

        all_chunks.append(pooled.cpu().numpy().astype(np.float32))

    embeddings = np.vstack(all_chunks).astype(np.float32)
    np.save(output_npy, embeddings)
    return embeddings


def build_id_maps(all_item_ids: np.ndarray) -> tuple[dict[int, int], dict[int, int]]:
    item_to_reduced = {int(item): idx + 1 for idx, item in enumerate(all_item_ids.tolist())}
    reduced_to_item = {idx + 1: int(item) for idx, item in enumerate(all_item_ids.tolist())}
    return item_to_reduced, reduced_to_item


def to_reduced_sequence(seq: np.ndarray, item_to_reduced: dict[int, int]) -> list[int]:
    return [item_to_reduced[int(i)] for i in seq if int(i) in item_to_reduced]


def build_train_examples(train_df: pd.DataFrame, item_to_reduced: dict[int, int], seq_len: int) -> tuple[np.ndarray, np.ndarray]:
    x_rows: list[np.ndarray] = []
    y_rows: list[int] = []

    for _, cur in train_df.groupby("SessionId"):
        items = to_reduced_sequence(cur["ItemId"].to_numpy(dtype=int), item_to_reduced)
        if len(items) < 2:
            continue
        for t in range(1, len(items)):
            prompt = items[max(0, t - seq_len):t]
            target = items[t]
            x = np.zeros(seq_len, dtype=np.int32)
            x[-len(prompt):] = np.array(prompt, dtype=np.int32)
            x_rows.append(x)
            y_rows.append(target - 1)  # logits are for item indices 1..num_items
    return np.array(x_rows, dtype=np.int32), np.array(y_rows, dtype=np.int32)


def build_predict_array(prompts: dict[int, np.ndarray], item_to_reduced: dict[int, int], seq_len: int) -> tuple[list[int], np.ndarray, list[list[int]]]:
    sids: list[int] = []
    x_rows: list[np.ndarray] = []
    seen_rows: list[list[int]] = []

    for sid, prompt in prompts.items():
        red = to_reduced_sequence(prompt, item_to_reduced)
        if not red:
            continue
        arr = np.zeros(seq_len, dtype=np.int32)
        arr[-len(red[-seq_len:]):] = np.array(red[-seq_len:], dtype=np.int32)
        sids.append(int(sid))
        x_rows.append(arr)
        seen_rows.append(red)

    if not x_rows:
        return [], np.empty((0, seq_len), dtype=np.int32), []
    return sids, np.array(x_rows, dtype=np.int32), seen_rows


def build_sasrec_like_model(num_items: int, seq_len: int, emb_dim: int, pretrained_item_embeddings: np.ndarray | None = None) -> keras.Model:
    inp = keras.Input(shape=(seq_len,), dtype="int32")

    item_emb_layer = keras.layers.Embedding(
        input_dim=num_items + 1,
        output_dim=emb_dim,
        mask_zero=True,
        name="item_emb",
    )

    x = item_emb_layer(inp)

    pos_idx = tf.range(start=0, limit=seq_len, delta=1)
    pos_emb_layer = keras.layers.Embedding(input_dim=seq_len, output_dim=emb_dim, name="pos_emb")
    pos_emb = pos_emb_layer(pos_idx)
    x = x + pos_emb

    key_dim = max(1, emb_dim // 2)
    attn = keras.layers.MultiHeadAttention(num_heads=2, key_dim=key_dim, dropout=0.2)
    attn_out = attn(x, x, use_causal_mask=True)
    x = keras.layers.LayerNormalization(epsilon=1e-6)(x + attn_out)

    ffn = keras.Sequential(
        [
            keras.layers.Dense(emb_dim * 2, activation="relu"),
            keras.layers.Dropout(0.2),
            keras.layers.Dense(emb_dim),
        ]
    )
    ffn_out = ffn(x)
    x = keras.layers.LayerNormalization(epsilon=1e-6)(x + ffn_out)

    non_zero = tf.cast(tf.not_equal(inp, 0), tf.int32)
    last_idx = tf.maximum(tf.reduce_sum(non_zero, axis=1) - 1, 0)
    batch_idx = tf.range(tf.shape(inp)[0], dtype=tf.int32)
    gather_idx = tf.stack([batch_idx, last_idx], axis=1)
    last_hidden = tf.gather_nd(x, gather_idx)

    item_table = item_emb_layer.embeddings[1:]  # [num_items, emb_dim]
    logits = keras.ops.matmul(last_hidden, keras.ops.transpose(item_table))

    model = keras.Model(inputs=inp, outputs=logits)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    )

    if pretrained_item_embeddings is not None:
        if pretrained_item_embeddings.shape != (num_items, emb_dim):
            raise ValueError(
                f"Expected pretrained embeddings shape {(num_items, emb_dim)}, got {pretrained_item_embeddings.shape}"
            )
        weights = item_emb_layer.get_weights()
        table = weights[0]
        table[1:, :] = pretrained_item_embeddings.astype(np.float32)
        item_emb_layer.set_weights([table])

    return model


def predict_topk(model: keras.Model, prompts: dict[int, np.ndarray], item_to_reduced: dict[int, int], reduced_to_item: dict[int, int], seq_len: int, top_k: int) -> dict[int, np.ndarray]:
    sids, x_arr, seen_rows = build_predict_array(prompts, item_to_reduced, seq_len)
    if len(sids) == 0:
        return {}

    logits = model.predict(x_arr, verbose=0, batch_size=1024)  # [B, num_items]
    preds: dict[int, np.ndarray] = {}

    for i, sid in enumerate(sids):
        scores = logits[i].astype(np.float64)
        seen = set(seen_rows[i])
        if seen:
            seen_idx = [r - 1 for r in seen if r > 0 and (r - 1) < len(scores)]
            scores[np.array(seen_idx, dtype=int)] = -1e12

        order = np.argsort(scores)[::-1][:top_k]
        rec_reduced = [int(o) + 1 for o in order]
        rec_orig = [reduced_to_item[r] for r in rec_reduced if r in reduced_to_item]
        preds[sid] = np.array(rec_orig, dtype=int)

    return preds


def predict_popular(prompts: dict[int, np.ndarray], train_counts: dict[int, int], top_k: int) -> dict[int, np.ndarray]:
    popular_items = [int(i) for i, _ in sorted(train_counts.items(), key=lambda kv: kv[1], reverse=True)]
    out: dict[int, np.ndarray] = {}
    for sid, prompt in prompts.items():
        seen = set(int(i) for i in prompt)
        recs = [i for i in popular_items if i not in seen][:top_k]
        out[int(sid)] = np.array(recs, dtype=int)
    return out


def session_embedding(prompt: np.ndarray, item_to_idx: dict[int, int], emb_norm: np.ndarray, decay: float = 1.0) -> np.ndarray | None:
    idxs = [item_to_idx[int(i)] for i in prompt if int(i) in item_to_idx]
    if not idxs:
        return None
    vecs = emb_norm[idxs]
    if len(vecs) == 1:
        return vecs[0]
    if decay == 1.0:
        out = vecs.mean(axis=0)
    else:
        n = len(vecs)
        weights = np.array([decay ** (n - i - 1) for i in range(n)], dtype=np.float32)
        weights = weights / np.sum(weights)
        out = np.average(vecs, axis=0, weights=weights)
    norm = np.linalg.norm(out)
    return out if norm == 0 else out / norm


def debias_weights(counts: np.ndarray, alpha: float, variant: str, ranks: np.ndarray, median_count: float) -> np.ndarray:
    c = np.maximum(counts.astype(np.float64), 1.0)
    if variant == "inverse":
        return 1.0 / np.power(c, alpha)
    if variant == "log_norm":
        return 1.0 / np.power(1.0 + np.log(c), alpha)
    if variant == "rank":
        return 1.0 / np.power(np.maximum(ranks.astype(np.float64), 1.0), alpha)
    if variant == "sigmoid":
        x = -alpha * ((c / max(median_count, 1.0)) - 1.0)
        return 1.0 / (1.0 + np.exp(-x))
    raise ValueError(f"Unknown variant: {variant}")


def rerank_popdebias(base_candidates: dict[int, np.ndarray], prompts: dict[int, np.ndarray], item_to_idx: dict[int, int], emb_norm: np.ndarray, item_counts: dict[int, int], alpha: float, top_k: int, variant: str = "inverse") -> dict[int, np.ndarray]:
    sorted_pop = sorted(item_counts.items(), key=lambda kv: kv[1], reverse=True)
    rank_map = {int(item): idx + 1 for idx, (item, _) in enumerate(sorted_pop)}
    median_count = float(np.median(np.array(list(item_counts.values()), dtype=np.float64)))

    out: dict[int, np.ndarray] = {}
    for sid, cand in base_candidates.items():
        q = session_embedding(prompts[sid], item_to_idx, emb_norm, decay=1.0)
        if q is None:
            out[sid] = np.array(cand[:top_k], dtype=int)
            continue

        cand = np.array([int(i) for i in cand if int(i) in item_to_idx], dtype=int)
        if cand.size == 0:
            out[sid] = np.array([], dtype=int)
            continue

        idxs = np.array([item_to_idx[int(i)] for i in cand], dtype=int)
        sims = emb_norm[idxs] @ q
        counts = np.array([item_counts.get(int(i), 1) for i in cand], dtype=np.float64)
        ranks = np.array([rank_map.get(int(i), len(rank_map) + 1) for i in cand], dtype=np.float64)
        weights = debias_weights(counts, alpha, variant, ranks, median_count)
        scores = sims * weights

        seen = set(int(i) for i in prompts[sid])
        if seen:
            mask = np.array([i not in seen for i in cand], dtype=bool)
            cand = cand[mask]
            scores = scores[mask]
        if cand.size == 0:
            out[sid] = np.array([], dtype=int)
            continue

        order = np.argsort(scores)[::-1]
        out[sid] = cand[order][:top_k].astype(int)
    return out


def predict_cold_branch(prompts: dict[int, np.ndarray], all_item_ids: np.ndarray, item_to_idx: dict[int, int], emb_norm: np.ndarray, top_k: int, decay: float) -> dict[int, np.ndarray]:
    out: dict[int, np.ndarray] = {}
    for sid, prompt in prompts.items():
        q = session_embedding(prompt, item_to_idx, emb_norm, decay=decay)
        if q is None:
            out[sid] = all_item_ids[:top_k].astype(int)
            continue
        scores = emb_norm @ q
        seen_idxs = [item_to_idx[int(i)] for i in prompt if int(i) in item_to_idx]
        if seen_idxs:
            scores[np.array(seen_idxs, dtype=int)] = -1e12
        order = np.argsort(scores)[::-1][:top_k]
        out[sid] = all_item_ids[order].astype(int)
    return out


def route_coldbridge(warm_preds: dict[int, np.ndarray], cold_preds: dict[int, np.ndarray], prompts: dict[int, np.ndarray], tau: int, top_k: int) -> dict[int, np.ndarray]:
    out: dict[int, np.ndarray] = {}
    for sid, prompt in prompts.items():
        out[sid] = (cold_preds[sid] if len(prompt) <= tau else warm_preds[sid])[:top_k]
    return out


def evaluate_at_k(preds: dict[int, np.ndarray], gts: dict[int, np.ndarray], k: int) -> tuple[float, float]:
    hrs = []
    ndcgs = []
    for sid, gt in gts.items():
        target = int(gt[0])
        rec = [int(i) for i in preds.get(sid, np.array([], dtype=int))[:k]]
        if target in rec:
            hrs.append(1.0)
            rank = rec.index(target) + 1
            ndcgs.append(1.0 / math.log2(rank + 1.0))
        else:
            hrs.append(0.0)
            ndcgs.append(0.0)
    return float(np.mean(ndcgs) if ndcgs else 0.0), float(np.mean(hrs) if hrs else 0.0)


def long_tail_hr10(preds: dict[int, np.ndarray], gts: dict[int, np.ndarray], item_counts: dict[int, int], threshold: int) -> float:
    hits = 0
    total = 0
    for sid, gt in gts.items():
        target = int(gt[0])
        if item_counts.get(target, 0) >= threshold:
            continue
        total += 1
        rec = [int(i) for i in preds.get(sid, np.array([], dtype=int))[:10]]
        if target in rec:
            hits += 1
    return float(hits / total) if total > 0 else 0.0


def catalog_coverage10(preds: dict[int, np.ndarray], num_items: int) -> float:
    used = set()
    for rec in preds.values():
        used.update(int(i) for i in rec[:10])
    return float(len(used) / num_items) if num_items > 0 else 0.0


def serendipity10(preds: dict[int, np.ndarray], gts: dict[int, np.ndarray], expected_popular_top10: list[int]) -> float:
    vals = []
    expected = set(expected_popular_top10)
    for sid, gt in gts.items():
        target = int(gt[0])
        rec = [int(i) for i in preds.get(sid, np.array([], dtype=int))[:10]]
        vals.append(1.0 if (target in rec and target not in expected) else 0.0)
    return float(np.mean(vals) if vals else 0.0)


def ild10(preds: dict[int, np.ndarray], item_to_idx: dict[int, int], emb_norm: np.ndarray) -> float:
    vals = []
    for rec in preds.values():
        ids = [int(i) for i in rec[:10] if int(i) in item_to_idx]
        if len(ids) < 2:
            continue
        idxs = np.array([item_to_idx[i] for i in ids], dtype=int)
        vecs = emb_norm[idxs]
        sims = vecs @ vecs.T
        iu = np.triu_indices(len(ids), k=1)
        if len(iu[0]) == 0:
            continue
        vals.append(float(np.mean(1.0 - sims[iu])))
    return float(np.mean(vals) if vals else 0.0)


def segmented_hr10(preds: dict[int, np.ndarray], gts: dict[int, np.ndarray], lengths: dict[int, int], cold_thr: int = 5) -> tuple[float, float]:
    cold_vals = []
    warm_vals = []
    for sid, gt in gts.items():
        target = int(gt[0])
        rec = [int(i) for i in preds.get(sid, np.array([], dtype=int))[:10]]
        hit = 1.0 if target in rec else 0.0
        if lengths.get(sid, 0) <= cold_thr:
            cold_vals.append(hit)
        else:
            warm_vals.append(hit)
    cold_hr = float(np.mean(cold_vals) if cold_vals else 0.0)
    warm_hr = float(np.mean(warm_vals) if warm_vals else 0.0)
    return cold_hr, warm_hr


def evaluate_predictions(model_name: str, preds: dict[int, np.ndarray], gts: dict[int, np.ndarray], item_counts: dict[int, int], num_items: int, emb_norm: np.ndarray, item_to_idx: dict[int, int], expected_pop_top10: list[int], alpha: float | None, tau: int | None, training_time: float, inference_time: float) -> dict[str, Any]:
    ndcg10, hr10 = evaluate_at_k(preds, gts, 10)
    ndcg20, hr20 = evaluate_at_k(preds, gts, 20)
    row = {
        "model_name": model_name,
        "alpha": alpha,
        "tau": tau,
        "NDCG@10": ndcg10,
        "NDCG@20": ndcg20,
        "HR@10": hr10,
        "HR@20": hr20,
        "LongTail_HR@10": long_tail_hr10(preds, gts, item_counts, LONG_TAIL_THRESHOLD),
        "CatalogCoverage": catalog_coverage10(preds, num_items),
        "Serendipity": serendipity10(preds, gts, expected_pop_top10),
        "ILD@10": ild10(preds, item_to_idx, emb_norm),
        "training_time_sec": float(training_time),
        "inference_time_sec": float(inference_time),
    }
    return row


def choose_best_joint(rows: list[dict[str, Any]]) -> dict[str, Any]:
    scored = []
    for row in rows:
        joint = 0.7 * float(row["NDCG@10"]) + 0.3 * float(row["LongTail_HR@10"])
        scored.append((joint, row))
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[0][1]


def save_best_model(best_row: dict[str, Any], baseline_row: dict[str, Any] | None, output_path: Path) -> None:
    baseline_ndcg = float(baseline_row["NDCG@10"]) if baseline_row is not None else None
    best_ndcg = float(best_row["NDCG@10"])

    if baseline_ndcg and baseline_ndcg > 0:
        delta = ((best_ndcg - baseline_ndcg) / baseline_ndcg) * 100.0
        delta_text = f"{delta:+.2f}%"
        base_text = f"{baseline_ndcg:.6f}"
    else:
        delta_text = "N/A"
        base_text = "N/A"

    lines = [
        f"BEST MODEL: {best_row['model_name']}",
        f"Best Alpha: {best_row.get('alpha')}",
        f"Best Tau:   {best_row.get('tau')}",
        f"NDCG@10:    {best_ndcg:.6f} (vs baseline: {base_text}, delta: {delta_text})",
        f"HR@10:      {float(best_row['HR@10']):.6f}",
        f"LT-HR@10:   {float(best_row['LongTail_HR@10']):.6f}",
        f"ILD@10:     {float(best_row['ILD@10']):.6f}",
    ]
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text("\n".join(lines) + "\n", encoding="utf-8")

In [ ]:
# Keras-safe override for the sequence model used below.
# Uses GRU blocks instead of MultiHeadAttention to avoid Kaggle/Keras einsum issues.
def build_sasrec_like_model(num_items: int, seq_len: int, emb_dim: int, pretrained_item_embeddings: np.ndarray | None = None) -> keras.Model:
    inp = keras.Input(shape=(seq_len,), dtype="int32")

    item_emb_layer = keras.layers.Embedding(
        input_dim=num_items + 1,
        output_dim=emb_dim,
        mask_zero=True,
        name="item_emb",
    )
    x = item_emb_layer(inp)

    pos_idx = tf.range(start=0, limit=seq_len, delta=1)
    pos_emb_layer = keras.layers.Embedding(input_dim=seq_len, output_dim=emb_dim, name="pos_emb")
    x = x + pos_emb_layer(pos_idx)
    x = keras.layers.Dropout(0.2)(x)

    x = keras.layers.GRU(emb_dim, return_sequences=True, dropout=0.2)(x)
    x = keras.layers.LayerNormalization(epsilon=1e-6)(x)
    x = keras.layers.GRU(emb_dim, return_sequences=False, dropout=0.2)(x)
    x = keras.layers.LayerNormalization(epsilon=1e-6)(x)

    # Tie output projection to item embedding table to avoid Dense-layer activation resolution bugs.
    item_table = item_emb_layer.embeddings[1:]  # [num_items, emb_dim]
    logits = keras.ops.matmul(x, keras.ops.transpose(item_table))

    model = keras.Model(inputs=inp, outputs=logits)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    )

    if pretrained_item_embeddings is not None:
        if pretrained_item_embeddings.shape != (num_items, emb_dim):
            raise ValueError(
                f"Expected pretrained embeddings shape {(num_items, emb_dim)}, got {pretrained_item_embeddings.shape}"
            )
        table = item_emb_layer.get_weights()[0]
        table[1:, :] = pretrained_item_embeddings.astype(np.float32)
        item_emb_layer.set_weights([table])

    return model

In [ ]:
# v3 utilities: diagnostics + stronger ColdBridge + safe debias + hybrid
import itertools

RUN_MODE = "v3_fast"  # set "full" for exhaustive search
RANDOM_SEED = SEED
P_CORE = MIN_SESSION_LEN

V1_EXPECTED_BGE = {"NDCG@10": 0.04612, "HR@10": 0.08534}
V1_REPRO_MIN = {"NDCG@10": 0.04380, "HR@10": 0.08110}

CORE_TARGETS_DEFAULT = {
    "LLM2SASRec": {"NDCG@10": 0.05994, "HR@10": 0.12472, "CatalogCoverage": 0.23228},
    "BERT4Rec": {"NDCG@10": 0.05594, "HR@10": 0.10913, "CatalogCoverage": 0.24300},
    "SASRec": {"NDCG@10": 0.07303, "HR@10": 0.14254, "CatalogCoverage": 0.36152},
}


SENTENCE_EMBEDDING_CONFIG = {
    "gte-large": {
        "model_name": "thenlper/gte-large",
        "prefix": "",
        "file_name": "item_embeddings_gte_large.npy",
        "trust_remote_code": False,
    },
    "e5-large-v2": {
        "model_name": "intfloat/e5-large-v2",
        "prefix": "passage: ",
        "file_name": "item_embeddings_e5_large_v2.npy",
        "trust_remote_code": False,
    },
    "nomic-embed-text-v1.5": {
        "model_name": "nomic-ai/nomic-embed-text-v1.5",
        "prefix": "search_document: ",
        "file_name": "item_embeddings_nomic_v15.npy",
        "trust_remote_code": True,
    },
}


def _safe_float(x: Any, default: float = float("nan")) -> float:
    try:
        return float(x)
    except Exception:
        return default


def load_core_targets() -> dict[str, dict[str, float]]:
    candidate_paths = [
        Path("kaggle/result/core_model_results_ml100k.csv"),
        Path("../result/core_model_results_ml100k.csv"),
        Path("/kaggle/working/LLM-Sequential-Recommendation/kaggle/result/core_model_results_ml100k.csv"),
    ]

    for p in candidate_paths:
        if not p.exists():
            continue

        try:
            df = pd.read_csv(p)
        except Exception:
            continue

        parsed: dict[str, dict[str, float]] = {}
        for _, row in df.iterrows():
            model_name = str(row.get("Model", row.get("RequestedModel", ""))).strip()
            if not model_name:
                continue
            parsed[model_name] = {
                "NDCG@10": _safe_float(row.get("NDCG@10", np.nan)),
                "HR@10": _safe_float(row.get("HitRate@10", row.get("HR@10", np.nan))),
                "CatalogCoverage": _safe_float(row.get("Catalog coverage@10", row.get("CatalogCoverage", np.nan))),
            }

        needed = ["LLM2SASRec", "BERT4Rec", "SASRec"]
        if all(k in parsed for k in needed):
            return {k: parsed[k] for k in needed}

    return dict(CORE_TARGETS_DEFAULT)


def pipeline_diagnostic(
    interactions: pd.DataFrame,
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
    test_prompts: dict[int, np.ndarray],
    all_item_ids: np.ndarray,
) -> None:
    print("=== PIPELINE DIAGNOSTIC ===")
    print(f"Total users:        {interactions['UserId'].nunique()}")
    print(f"Total items:        {len(all_item_ids)}")
    print(f"Train sessions:     {train_df['SessionId'].nunique()}")
    print(f"Val sessions:       {val_df['SessionId'].nunique()}")
    print(f"Test sessions:      {test_df['SessionId'].nunique()}")
    print(f"Train interactions: {len(train_df)}")
    print(f"Test interactions:  {len(test_df)}")

    if test_prompts:
        avg_len = float(np.mean([len(v) for v in test_prompts.values()]))
    else:
        avg_len = 0.0

    n_train = train_df["SessionId"].nunique()
    n_val = val_df["SessionId"].nunique()
    n_test = test_df["SessionId"].nunique()
    ratio = (n_test / max(n_train + n_val + n_test, 1))

    print(f"Avg test seq len:   {avg_len:.2f}")
    print(f"Test split ratio:   {ratio:.3f}")
    print(f"Random seed used:   {RANDOM_SEED}")
    print(f"p-core value:       {P_CORE}")
    print()


def train_with_callbacks(
    model: keras.Model,
    x: np.ndarray,
    y: np.ndarray,
    max_epochs: int,
    batch_size: int,
    validation_split: float = 0.1,
    early_stop_patience: int = 12,
    lr_patience: int = 4,
) -> tuple[float, int, str]:
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=early_stop_patience,
            restore_best_weights=True,
            min_delta=1e-4,
            verbose=0,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=lr_patience,
            min_lr=1e-5,
            verbose=0,
        ),
    ]

    t0 = time.perf_counter()
    history = model.fit(
        x,
        y,
        epochs=max_epochs,
        batch_size=batch_size,
        validation_split=validation_split,
        callbacks=callbacks,
        verbose=0,
        shuffle=True,
    )
    train_time = time.perf_counter() - t0

    val_loss = history.history.get("val_loss", [])
    train_loss = history.history.get("loss", [])
    tracked = val_loss if len(val_loss) > 0 else train_loss
    best_epoch = int(np.argmin(np.array(tracked, dtype=np.float64)) + 1) if tracked else int(max_epochs)

    return float(train_time), best_epoch, "ReduceLROnPlateau+EarlyStopping"


def encode_sentence_transformer_model(
    embedding_key: str,
    item_ids: np.ndarray,
    item_text_df: pd.DataFrame,
    output_npy: Path,
) -> np.ndarray:
    cfg = SENTENCE_EMBEDDING_CONFIG[embedding_key]
    output_npy.parent.mkdir(parents=True, exist_ok=True)

    if output_npy.exists():
        cached = np.load(output_npy)
        if cached.shape[0] == len(item_ids):
            return normalize_rows(cached.astype(np.float32))

    from sentence_transformers import SentenceTransformer

    text_lookup = item_text_df.set_index("ItemId")["item_text"].to_dict()
    texts = [text_lookup.get(int(i), f"Movie item {int(i)}.") for i in item_ids]

    prefix = str(cfg["prefix"])
    if prefix:
        texts = [prefix + t for t in texts]

    model = SentenceTransformer(
        str(cfg["model_name"]),
        trust_remote_code=bool(cfg["trust_remote_code"]),
    )
    embs = model.encode(
        texts,
        batch_size=128,
        normalize_embeddings=True,
        show_progress_bar=False,
    )

    embs = np.asarray(embs, dtype=np.float32)
    np.save(output_npy, embs)
    return normalize_rows(embs)


def get_item_embeddings(
    embedding_key: str,
    item_ids: np.ndarray,
    item_text_df: pd.DataFrame,
    emb_dir: Path,
) -> np.ndarray:
    if embedding_key == "bge-m3":
        bge_path = emb_dir / "item_embeddings_bge_m3.npy"
        return normalize_rows(encode_bge_cpu(item_ids, item_text_df, bge_path).astype(np.float32))

    if embedding_key not in SENTENCE_EMBEDDING_CONFIG:
        raise ValueError(f"Unknown embedding key: {embedding_key}")

    out_path = emb_dir / str(SENTENCE_EMBEDDING_CONFIG[embedding_key]["file_name"])
    return encode_sentence_transformer_model(embedding_key, item_ids, item_text_df, out_path)


def predict_topk_with_scores(
    model: keras.Model,
    prompts: dict[int, np.ndarray],
    item_to_reduced: dict[int, int],
    reduced_to_item: dict[int, int],
    seq_len: int,
    top_k: int,
) -> tuple[dict[int, np.ndarray], dict[int, np.ndarray]]:
    sids, x_arr, seen_rows = build_predict_array(prompts, item_to_reduced, seq_len)
    if len(sids) == 0:
        return {}, {}

    logits = model.predict(x_arr, verbose=0, batch_size=1024)
    out_items: dict[int, np.ndarray] = {}
    out_scores: dict[int, np.ndarray] = {}

    for i, sid in enumerate(sids):
        scores = logits[i].astype(np.float64)
        seen = set(seen_rows[i])
        if seen:
            seen_idx = [r - 1 for r in seen if r > 0 and (r - 1) < len(scores)]
            if seen_idx:
                scores[np.array(seen_idx, dtype=int)] = -1e12

        order = np.argsort(scores)[::-1][:top_k]
        reduced_items = np.array([int(o) + 1 for o in order], dtype=int)
        orig_items = np.array([reduced_to_item[r] for r in reduced_items if r in reduced_to_item], dtype=int)
        out_items[sid] = orig_items
        out_scores[sid] = scores[order][: len(orig_items)]

    return out_items, out_scores


def topk_from_candidates_scores(
    candidates: dict[int, np.ndarray],
    scores: dict[int, np.ndarray],
    top_k: int,
) -> dict[int, np.ndarray]:
    out: dict[int, np.ndarray] = {}
    for sid, items in candidates.items():
        cur_items = np.asarray(items, dtype=int)
        cur_scores = np.asarray(scores.get(sid, np.zeros(len(cur_items), dtype=np.float64)), dtype=np.float64)
        if cur_items.size == 0:
            out[sid] = np.array([], dtype=int)
            continue
        order = np.argsort(cur_scores)[::-1][:top_k]
        out[sid] = cur_items[order].astype(int)
    return out


def print_step_status(
    step: int,
    model_name: str,
    row: dict[str, Any],
    core_targets: dict[str, dict[str, float]],
    passed: bool,
    fail_reason: str = "",
) -> None:
    ndcg10 = float(row.get("NDCG@10", np.nan))
    hr10 = float(row.get("HR@10", np.nan))
    cov = float(row.get("CatalogCoverage", np.nan))
    ild = float(row.get("ILD@10", np.nan))

    print(
        f"[STEP {step}] model={model_name} "
        f"NDCG@10={ndcg10:.6f} HR@10={hr10:.6f} Coverage={cov:.6f} ILD={ild:.6f}"
    )

    if passed:
        print("[STATUS] PASS")
    else:
        suffix = f" ({fail_reason})" if fail_reason else ""
        print(f"[STATUS] FAIL{suffix}")

    core = core_targets.get("LLM2SASRec", CORE_TARGETS_DEFAULT["LLM2SASRec"])
    core_ndcg = float(core.get("NDCG@10", np.nan))
    core_hr = float(core.get("HR@10", np.nan))

    ndcg_delta = float("nan") if not np.isfinite(core_ndcg) or core_ndcg == 0 else ((ndcg10 - core_ndcg) / core_ndcg) * 100.0
    hr_delta = float("nan") if not np.isfinite(core_hr) or core_hr == 0 else ((hr10 - core_hr) / core_hr) * 100.0
    print(f"[vs CORE LLM2SASRec] NDCG delta={ndcg_delta:+.2f}%  HR delta={hr_delta:+.2f}%")


def find_pareto_front(results_df: pd.DataFrame) -> np.ndarray:
    metrics = ["NDCG@10", "HR@10", "CatalogCoverage", "ILD@10"]
    vals = results_df[metrics].to_numpy(dtype=np.float64)
    n = len(vals)
    is_pareto = np.ones(n, dtype=bool)
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            if np.all(vals[j] >= vals[i]) and np.any(vals[j] > vals[i]):
                is_pareto[i] = False
                break
    return is_pareto


def build_item_count_array(all_item_ids: np.ndarray, item_counts: dict[int, int]) -> np.ndarray:
    return np.array([float(item_counts.get(int(i), 1)) for i in all_item_ids], dtype=np.float64)


def cold_branch_score(
    user_items: np.ndarray,
    all_item_ids: np.ndarray,
    item_to_idx: dict[int, int],
    emb_norm: np.ndarray,
    item_count_array: np.ndarray,
    decay: float = 0.8,
    alpha_debias: float = 0.1,
) -> np.ndarray | None:
    idxs = [item_to_idx[int(i)] for i in user_items if int(i) in item_to_idx]
    if not idxs:
        return None

    vecs = emb_norm[np.array(idxs, dtype=int)]
    n = len(vecs)

    weights = np.array([decay ** (n - k - 1) for k in range(n)], dtype=np.float64)
    weights = weights / np.sum(weights)

    session_emb = np.average(vecs, axis=0, weights=weights)
    norm = np.linalg.norm(session_emb)
    if norm > 0:
        session_emb = session_emb / norm

    scores = emb_norm @ session_emb

    if alpha_debias > 0:
        inv_freq = 1.0 / np.power(np.maximum(item_count_array, 1.0), alpha_debias)
        scores = scores * inv_freq

    return scores


def predict_cold_branch_v3(
    prompts: dict[int, np.ndarray],
    all_item_ids: np.ndarray,
    item_to_idx: dict[int, int],
    emb_norm: np.ndarray,
    item_count_array: np.ndarray,
    top_k: int,
    decay: float,
    alpha_debias: float,
) -> dict[int, np.ndarray]:
    out: dict[int, np.ndarray] = {}

    for sid, prompt in prompts.items():
        scores = cold_branch_score(
            user_items=prompt,
            all_item_ids=all_item_ids,
            item_to_idx=item_to_idx,
            emb_norm=emb_norm,
            item_count_array=item_count_array,
            decay=decay,
            alpha_debias=alpha_debias,
        )

        if scores is None:
            out[sid] = all_item_ids[:top_k].astype(int)
            continue

        seen_idxs = [item_to_idx[int(i)] for i in prompt if int(i) in item_to_idx]
        if seen_idxs:
            scores[np.array(seen_idxs, dtype=int)] = -1e12

        order = np.argsort(scores)[::-1][:top_k]
        out[sid] = all_item_ids[order].astype(int)

    return out


def slot_fill_debias(
    ranked_items: np.ndarray | list[int],
    item_counts: dict[int, int],
    top_k: int = 20,
    protected_top: int = 5,
    diversity_slots: int = 5,
    tail_threshold: int = 200,
) -> list[int]:
    ranked = [int(i) for i in ranked_items]
    final: list[int] = []

    for i in ranked[:protected_top]:
        if i not in final:
            final.append(i)

    remaining = [i for i in ranked if i not in final]
    tail_items = [i for i in remaining if item_counts.get(int(i), 0) < tail_threshold]
    head_items = [i for i in remaining if item_counts.get(int(i), 0) >= tail_threshold]

    slots = max(top_k - len(final), 0)
    n_tail_target = min(int(diversity_slots), len(tail_items), slots)

    combined: list[int] = []
    t_idx = 0
    h_idx = 0
    pos = 0

    while len(combined) < slots:
        use_tail = (pos % 3 == 0) and (t_idx < n_tail_target)

        if use_tail:
            combined.append(int(tail_items[t_idx]))
            t_idx += 1
        elif h_idx < len(head_items):
            combined.append(int(head_items[h_idx]))
            h_idx += 1
        elif t_idx < len(tail_items):
            combined.append(int(tail_items[t_idx]))
            t_idx += 1
        else:
            break

        pos += 1

    for i in remaining:
        if len(combined) >= slots:
            break
        if i not in combined:
            combined.append(int(i))

    final.extend(combined)
    return final[:top_k]


def llmseqsim_rank(
    prompt: np.ndarray,
    all_item_ids: np.ndarray,
    item_to_idx: dict[int, int],
    emb_norm: np.ndarray,
    top_n: int = 100,
) -> list[int]:
    q = session_embedding(prompt, item_to_idx, emb_norm, decay=1.0)
    if q is None:
        return [int(i) for i in all_item_ids[:top_n]]

    scores = emb_norm @ q
    seen_idxs = [item_to_idx[int(i)] for i in prompt if int(i) in item_to_idx]
    if seen_idxs:
        scores[np.array(seen_idxs, dtype=int)] = -1e12

    order = np.argsort(scores)[::-1][:top_n]
    return [int(all_item_ids[idx]) for idx in order]


def hybrid_position_ensemble(
    seq_recs: np.ndarray | list[int],
    llmseqsim_recs: np.ndarray | list[int],
    top_k: int = 20,
    seq_positions: int = 15,
) -> list[int]:
    final: list[int] = []

    for item in list(seq_recs)[:seq_positions]:
        i = int(item)
        if i not in final:
            final.append(i)

    for item in llmseqsim_recs:
        if len(final) >= top_k:
            break
        i = int(item)
        if i not in final:
            final.append(i)

    return final[:top_k]


def build_full_system_predictions(
    prompts: dict[int, np.ndarray],
    warm_candidates: dict[int, np.ndarray],
    warm_scores: dict[int, np.ndarray],
    all_item_ids: np.ndarray,
    item_to_idx: dict[int, int],
    emb_norm: np.ndarray,
    item_count_array: np.ndarray,
    item_counts: dict[int, int],
    tau: int,
    top_k: int,
    decay: float,
    alpha_debias: float,
    protected_top: int | None = None,
    diversity_slots: int | None = None,
    tail_threshold: int = 200,
    seq_positions: int | None = None,
) -> dict[int, np.ndarray]:
    cold_preds = predict_cold_branch_v3(
        prompts=prompts,
        all_item_ids=all_item_ids,
        item_to_idx=item_to_idx,
        emb_norm=emb_norm,
        item_count_array=item_count_array,
        top_k=top_k,
        decay=decay,
        alpha_debias=alpha_debias,
    )

    out: dict[int, np.ndarray] = {}

    for sid, prompt in prompts.items():
        if len(prompt) < int(tau):
            out[sid] = cold_preds.get(sid, np.array([], dtype=int))[:top_k].astype(int)
            continue

        cand_items = np.asarray(warm_candidates.get(sid, np.array([], dtype=int)), dtype=int)
        cand_scores = np.asarray(warm_scores.get(sid, np.array([], dtype=np.float64)), dtype=np.float64)

        if cand_items.size == 0:
            out[sid] = cold_preds.get(sid, np.array([], dtype=int))[:top_k].astype(int)
            continue

        if cand_scores.size != cand_items.size:
            cand_scores = np.zeros(cand_items.size, dtype=np.float64)

        order = np.argsort(cand_scores)[::-1]
        ranked = [int(i) for i in cand_items[order]]

        if protected_top is not None and diversity_slots is not None:
            ranked = slot_fill_debias(
                ranked_items=ranked,
                item_counts=item_counts,
                top_k=top_k,
                protected_top=int(protected_top),
                diversity_slots=int(diversity_slots),
                tail_threshold=int(tail_threshold),
            )
        else:
            ranked = ranked[:top_k]

        if seq_positions is not None:
            llmseq = llmseqsim_rank(
                prompt=prompt,
                all_item_ids=all_item_ids,
                item_to_idx=item_to_idx,
                emb_norm=emb_norm,
                top_n=max(top_k * 2, 100),
            )
            ranked = hybrid_position_ensemble(
                seq_recs=ranked,
                llmseqsim_recs=llmseq,
                top_k=top_k,
                seq_positions=int(seq_positions),
            )

        out[sid] = np.array(ranked[:top_k], dtype=int)

    return out


def make_extended_schema_cols() -> list[str]:
    return [
        "model_name",
        "alpha",
        "tau",
        "NDCG@10",
        "NDCG@20",
        "HR@10",
        "HR@20",
        "LongTail_HR@10",
        "CatalogCoverage",
        "Serendipity",
        "ILD@10",
        "training_time_sec",
        "inference_time_sec",
        "debias_variant",
        "cold_decay",
        "lambda_sim",
        "lambda_pop",
        "top_n_anchor",
        "steepness",
        "blend_type",
        "embedding_model",
        "protected_top",
        "diversity_slots",
        "backbone",
        "lr_schedule",
        "early_stop_epoch",
        "tail_threshold",
        "seq_positions",
    ]


def pct_delta(cur: float, ref: float) -> float:
    if not np.isfinite(ref) or ref == 0:
        return float("nan")
    return ((cur - ref) / ref) * 100.0

In [ ]:
import importlib.util
import json
import os
import sys
import time
from pathlib import Path

# Runtime overrides for Kaggle runs (optional).
run_mode_env = os.environ.get("NOVELTY_RUN_MODE", "").strip().lower()
if run_mode_env:
    if run_mode_env in {"v3_fast", "full"}:
        RUN_MODE = run_mode_env
    else:
        print(f"Unknown NOVELTY_RUN_MODE={run_mode_env!r}; keeping RUN_MODE={RUN_MODE}.")

ENFORCE_REPRO_GATE = os.environ.get("ENFORCE_REPRO_GATE", "1").strip().lower() not in {"0", "false", "no"}

# Ensure embedding extras are available if full sweep is requested at runtime.
if RUN_MODE == "full":
    ensure_package("sentence_transformers", "sentence-transformers")

_V1_REPRO_CACHE: dict[str, Any] | None = None


def _is_repo_root(candidate: Path) -> bool:
    return (candidate / "main" / "transformer" / "sasrec" / "sasrec_with_embeddings.py").exists()


def _candidate_repo_roots() -> list[Path]:
    raw: list[Path] = [
        Path.cwd(),
        *Path.cwd().parents,
        Path("/kaggle/working"),
        Path("/kaggle/input"),
        Path("/kaggle/working/LLM-Sequential-Recommendation"),
        Path("/kaggle/working/llm-sequential-recommendation"),
        Path("/kaggle/input/LLM-Sequential-Recommendation"),
        Path("/kaggle/input/llm-sequential-recommendation"),
    ]

    for parent in (Path("/kaggle/working"), Path("/kaggle/input")):
        if not parent.exists():
            continue
        for child in parent.glob("*"):
            raw.append(child)
            raw.append(child / "LLM-Sequential-Recommendation")
            raw.append(child / "llm-sequential-recommendation")

    out: list[Path] = []
    seen: set[str] = set()
    for p in raw:
        try:
            rp = p.resolve()
        except Exception:
            rp = p
        key = str(rp)
        if key in seen:
            continue
        seen.add(key)
        out.append(rp)

    return out


def _ensure_repo_root_on_syspath() -> Path | None:
    # If already importable, recover repo root from the resolved module location.
    spec = importlib.util.find_spec("main.transformer.sasrec.sasrec_with_embeddings")
    if spec is not None and spec.origin:
        origin = Path(spec.origin).resolve()
        for parent in [origin.parent, *origin.parents]:
            if _is_repo_root(parent):
                if str(parent) not in sys.path:
                    sys.path.insert(0, str(parent))
                return parent

    # Otherwise, scan likely Kaggle/workspace locations.
    for candidate in _candidate_repo_roots():
        if not candidate.exists():
            continue
        if _is_repo_root(candidate):
            if str(candidate) not in sys.path:
                sys.path.insert(0, str(candidate))
            return candidate

    return None


def _write_bge_embedding_csv(item_ids: np.ndarray, emb_norm: np.ndarray, out_path: Path) -> Path:
    out_path.parent.mkdir(parents=True, exist_ok=True)

    if out_path.exists():
        try:
            cur = pd.read_csv(out_path)
            if len(cur) == len(item_ids):
                return out_path
        except Exception:
            pass

    df = pd.DataFrame(
        {
            "ItemId": item_ids.astype(int),
            "embedding": [json.dumps([float(x) for x in row]) for row in emb_norm.astype(np.float32)],
            "class": 0,
        }
    )
    df.to_csv(out_path, index=False)
    return out_path


def _build_attention_repro_model(
    num_items: int,
    seq_len: int,
    emb_dim: int,
    pretrained_item_embeddings: np.ndarray,
) -> keras.Model:
    inp = keras.Input(shape=(seq_len,), dtype="int32")

    item_emb_layer = keras.layers.Embedding(
        input_dim=num_items + 1,
        output_dim=emb_dim,
        mask_zero=True,
        name="item_emb_repro",
    )
    x = item_emb_layer(inp)

    pos_idx = tf.range(start=0, limit=seq_len, delta=1)
    pos_emb_layer = keras.layers.Embedding(input_dim=seq_len, output_dim=emb_dim, name="pos_emb_repro")
    x = x + pos_emb_layer(pos_idx)

    key_dim = max(1, emb_dim // 2)
    attn = keras.layers.MultiHeadAttention(num_heads=2, key_dim=key_dim, dropout=0.2)
    attn_out = attn(x, x, use_causal_mask=True)
    x = keras.layers.LayerNormalization(epsilon=1e-6)(x + attn_out)

    ffn = keras.Sequential(
        [
            keras.layers.Dense(emb_dim * 2, activation="relu"),
            keras.layers.Dropout(0.2),
            keras.layers.Dense(emb_dim),
        ]
    )
    ffn_out = ffn(x)
    x = keras.layers.LayerNormalization(epsilon=1e-6)(x + ffn_out)

    non_zero = tf.cast(tf.not_equal(inp, 0), tf.int32)
    last_idx = tf.maximum(tf.reduce_sum(non_zero, axis=1) - 1, 0)
    batch_idx = tf.range(tf.shape(inp)[0], dtype=tf.int32)
    gather_idx = tf.stack([batch_idx, last_idx], axis=1)
    last_hidden = tf.gather_nd(x, gather_idx)

    item_table = item_emb_layer.embeddings[1:]
    logits = keras.ops.matmul(last_hidden, keras.ops.transpose(item_table))

    model = keras.Model(inputs=inp, outputs=logits)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    )

    if pretrained_item_embeddings.shape != (num_items, emb_dim):
        raise ValueError(
            f"Expected pretrained embeddings shape {(num_items, emb_dim)}, got {pretrained_item_embeddings.shape}"
        )

    table = item_emb_layer.get_weights()[0]
    table[1:, :] = pretrained_item_embeddings.astype(np.float32)
    item_emb_layer.set_weights([table])

    return model


def _run_internal_repro_fallback() -> dict[str, Any] | None:
    required = [
        "x_train",
        "y_train",
        "test_prompts",
        "item_to_reduced",
        "reduced_to_item",
        "SEED",
    ]
    missing = [name for name in required if name not in globals()]
    if missing:
        print(f"[REPAIR] Internal fallback missing globals: {missing}")
        return None

    try:
        bge_reduced = normalize_rows(PCA(n_components=EMB_DIM, random_state=SEED).fit_transform(bge_embeddings).astype(np.float32))

        model = _build_attention_repro_model(
            num_items=num_items,
            seq_len=SEQ_LEN,
            emb_dim=EMB_DIM,
            pretrained_item_embeddings=bge_reduced,
        )

        repro_epochs = max(10, int(globals().get("NUM_EPOCHS", 4)) * 3)

        t0 = time.perf_counter()
        model.fit(x_train, y_train, epochs=repro_epochs, batch_size=BATCH_SIZE, verbose=0)
        train_time = time.perf_counter() - t0

        t0 = time.perf_counter()
        test_candidates, test_scores = predict_topk_with_scores(
            model,
            test_prompts,
            item_to_reduced,
            reduced_to_item,
            SEQ_LEN,
            CANDIDATE_K,
        )
        infer_time = time.perf_counter() - t0

        test_pred = topk_from_candidates_scores(test_candidates, test_scores, TOP_K)

        repaired_row = evaluate_predictions(
            model_name="BGE2SASRec-v3-repro",
            preds=test_pred,
            gts=test_gts,
            item_counts=item_counts,
            num_items=num_items,
            emb_norm=bge_embeddings,
            item_to_idx=item_to_idx,
            expected_pop_top10=expected_pop_top10,
            alpha=np.nan,
            tau=np.nan,
            training_time=train_time,
            inference_time=infer_time,
        )

        return repaired_row
    except Exception as exc:
        print(f"[REPAIR] Internal fallback failed: {exc}")
        return None


def _run_v1_repro_fallback() -> dict[str, Any] | None:
    global _V1_REPRO_CACHE

    if _V1_REPRO_CACHE is not None:
        return dict(_V1_REPRO_CACHE)

    required_globals = [
        "train_df",
        "test_prompts",
        "test_gts",
        "all_item_ids",
        "bge_embeddings",
        "item_counts",
        "num_items",
        "item_to_idx",
        "expected_pop_top10",
        "EMB_DIR",
        "SEQ_LEN",
        "EMB_DIM",
        "LEARNING_RATE",
        "BATCH_SIZE",
        "CANDIDATE_K",
        "TOP_K",
    ]
    missing = [name for name in required_globals if name not in globals()]
    if missing:
        print(f"[REPAIR] Missing globals for Step-1 fallback: {missing}")
        return None

    repo_root = _ensure_repo_root_on_syspath()
    if repo_root is None:
        print("[REPAIR] Could not locate repository root containing main/. Trying internal fallback.")
        repaired_row = _run_internal_repro_fallback()
        if repaired_row is not None:
            _V1_REPRO_CACHE = dict(repaired_row)
            return dict(_V1_REPRO_CACHE)
        return None

    print(f"[REPAIR] Using repo root: {repo_root}")

    try:
        from main.transformer.sasrec.sasrec_with_embeddings import SASRecWithEmbeddings
    except Exception as exc:
        print(f"[REPAIR] Failed to import SASRecWithEmbeddings: {exc}. Trying internal fallback.")
        repaired_row = _run_internal_repro_fallback()
        if repaired_row is not None:
            _V1_REPRO_CACHE = dict(repaired_row)
            return dict(_V1_REPRO_CACHE)
        return None

    emb_csv_path = _write_bge_embedding_csv(
        item_ids=all_item_ids,
        emb_norm=bge_embeddings,
        out_path=EMB_DIR / "ml100k_item_embeddings_bge_m3.csv",
    )

    num_epochs = max(6, int(globals().get("NUM_EPOCHS", 6)))

    model = SASRecWithEmbeddings(
        product_embeddings_location=str(emb_csv_path),
        red_method="PCA",
        red_params={},
        N=SEQ_LEN,
        L=1,
        h=2,
        emb_dim=EMB_DIM,
        drop_rate=0.2,
        activation="relu",
        optimizer_kwargs={"learning_rate": LEARNING_RATE, "weight_decay": 1e-4},
        transformer_layer_kwargs={"layout": "NFDR"},
        cores=2,
        is_verbose=False,
        num_epochs=num_epochs,
        fit_batch_size=BATCH_SIZE,
        pred_batch_size=4096,
        train_val_fraction=0.1,
        early_stopping_patience=1,
        pred_seen=False,
    )

    t0 = time.perf_counter()
    model.train(train_df)
    train_time = time.perf_counter() - t0

    t0 = time.perf_counter()
    test_candidates = model.predict(test_prompts, top_k=CANDIDATE_K)
    infer_time = time.perf_counter() - t0

    test_pred = {sid: np.asarray(rec[:TOP_K], dtype=int) for sid, rec in test_candidates.items()}

    repaired_row = evaluate_predictions(
        model_name="BGE2SASRec-v3-repro",
        preds=test_pred,
        gts=test_gts,
        item_counts=item_counts,
        num_items=num_items,
        emb_norm=bge_embeddings,
        item_to_idx=item_to_idx,
        expected_pop_top10=expected_pop_top10,
        alpha=np.nan,
        tau=np.nan,
        training_time=train_time,
        inference_time=infer_time,
    )

    _V1_REPRO_CACHE = dict(repaired_row)
    return dict(_V1_REPRO_CACHE)


# Wrap step logging once to enforce the Step-1 reproduction gate when requested.
if "_ORIG_PRINT_STEP_STATUS" not in globals():
    _ORIG_PRINT_STEP_STATUS = print_step_status


def print_step_status(step, model_name, row, core_targets, passed, fail_reason=""):
    effective_passed = bool(passed)
    effective_reason = str(fail_reason) if fail_reason else ""

    if step == 1 and not effective_passed:
        repaired_row = _run_v1_repro_fallback()
        if repaired_row is not None:
            row.update(repaired_row)

            repro_reasons = []
            if float(row.get("NDCG@10", 0.0)) < V1_REPRO_MIN["NDCG@10"]:
                repro_reasons.append("NDCG@10 below v1 reproduction floor")
            if float(row.get("HR@10", 0.0)) < V1_REPRO_MIN["HR@10"]:
                repro_reasons.append("HR@10 below v1 reproduction floor")

            effective_passed = len(repro_reasons) == 0
            effective_reason = "; ".join(repro_reasons)

            if effective_passed:
                print("[REPAIR] Step-1 parity restored via fallback path.")
            else:
                print("[REPAIR] Step-1 fallback ran but is still below the v1 floor.")
        else:
            print("[REPAIR] Step-1 fallback unavailable; keeping current Step-1 result.")

    _ORIG_PRINT_STEP_STATUS(step, model_name, row, core_targets, effective_passed, effective_reason)

    if step == 1 and (not effective_passed) and ENFORCE_REPRO_GATE:
        reason = effective_reason or "reproduction metrics below floor"
        raise RuntimeError(
            "Step-1 reproduction gate failed; fix split/config before continuing: " + reason
        )


print(f"Runtime config -> RUN_MODE={RUN_MODE}, ENFORCE_REPRO_GATE={ENFORCE_REPRO_GATE}")

In [ ]:
import itertools

# 1) Data preparation
udata_path, uitem_path = download_ml100k(DATA_DIR, DATASET_URL)
interactions = load_interactions(udata_path)
sessions_df = sessionize(interactions, SESSION_GAP_MINUTES, MIN_SESSION_LEN)
train_df, val_df, test_df = temporal_split(sessions_df, VAL_FRAC, TEST_FRAC)

train_prompts, train_gts, train_lengths = build_cases(train_df)
val_prompts, val_gts, val_lengths = build_cases(val_df)
test_prompts, test_gts, test_lengths = build_cases(test_df)

# 2) Metadata + embeddings + mappings
item_metadata = load_item_metadata(uitem_path)
all_item_ids = np.array(sorted(sessions_df["ItemId"].unique().tolist()), dtype=int)
num_items = len(all_item_ids)

# Base BGE embeddings are always computed first for repro checks.
bge_embeddings = get_item_embeddings("bge-m3", all_item_ids, item_metadata, EMB_DIR)

item_to_idx = {int(item): i for i, item in enumerate(all_item_ids.tolist())}
item_to_reduced, reduced_to_item = build_id_maps(all_item_ids)
x_train, y_train = build_train_examples(train_df, item_to_reduced, SEQ_LEN)

if x_train.size == 0:
    raise RuntimeError("No train examples were generated. Check split/sessionization settings.")

# 3) Pipeline diagnostic (mandatory)
pipeline_diagnostic(
    interactions=interactions,
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    test_prompts=test_prompts,
    all_item_ids=all_item_ids,
)

core_targets = load_core_targets()
print("Loaded core targets:", core_targets)

item_counts = train_df["ItemId"].value_counts().to_dict()
item_count_array = build_item_count_array(all_item_ids, item_counts)
popular_sorted = [int(i) for i, _ in sorted(item_counts.items(), key=lambda kv: kv[1], reverse=True)]
expected_pop_top10 = popular_sorted[:10]

results_rows: list[dict[str, Any]] = []
ablation_records: list[dict[str, Any]] = []


def apply_meta(row: dict[str, Any], **meta: Any) -> dict[str, Any]:
    out = dict(row)
    out.update(meta)
    return out


def with_v3_meta(row: dict[str, Any], **meta: Any) -> dict[str, Any]:
    defaults = {
        "debias_variant": np.nan,
        "cold_decay": np.nan,
        "lambda_sim": np.nan,
        "lambda_pop": np.nan,
        "top_n_anchor": np.nan,
        "steepness": np.nan,
        "blend_type": "baseline",
        "embedding_model": np.nan,
        "protected_top": np.nan,
        "diversity_slots": np.nan,
        "backbone": np.nan,
        "lr_schedule": np.nan,
        "early_stop_epoch": np.nan,
        "tail_threshold": np.nan,
        "seq_positions": np.nan,
    }
    defaults.update(meta)
    return apply_meta(row, **defaults)


# Step 0: v1 reference row for ablation table
ablation_records.append(
    {
        "step": 0,
        "model": "BGE2SASRec (v1 reference)",
        "components": "reference baseline",
        "NDCG@10": float(V1_EXPECTED_BGE["NDCG@10"]),
        "HR@10": float(V1_EXPECTED_BGE["HR@10"]),
    }
)


# MostPopular (kept for sanity)
t0 = time.perf_counter()
popular_preds_test = predict_popular(test_prompts, item_counts, TOP_K)
pop_inf_time = time.perf_counter() - t0
row_pop = evaluate_predictions(
    model_name="MostPopular",
    preds=popular_preds_test,
    gts=test_gts,
    item_counts=item_counts,
    num_items=num_items,
    emb_norm=bge_embeddings,
    item_to_idx=item_to_idx,
    expected_pop_top10=expected_pop_top10,
    alpha=np.nan,
    tau=np.nan,
    training_time=0.0,
    inference_time=pop_inf_time,
)
row_pop = with_v3_meta(row_pop, backbone="Popularity", blend_type="baseline", embedding_model="none")
results_rows.append(row_pop)


# Step 1: Reproduce v1-like BGE2SASRec baseline
pca_bge = PCA(n_components=EMB_DIM, random_state=SEED)
bge_reduced = normalize_rows(pca_bge.fit_transform(bge_embeddings).astype(np.float32))

bge2sas_repro = build_sasrec_like_model(
    num_items=num_items,
    seq_len=SEQ_LEN,
    emb_dim=EMB_DIM,
    pretrained_item_embeddings=bge_reduced,
)

t0 = time.perf_counter()
bge2sas_repro.fit(x_train, y_train, epochs=NUM_EPOCHS, batch_size=BATCH_SIZE, verbose=0)
repro_train_time = time.perf_counter() - t0

t0 = time.perf_counter()
repro_val_cands, repro_val_scores = predict_topk_with_scores(
    bge2sas_repro, val_prompts, item_to_reduced, reduced_to_item, SEQ_LEN, CANDIDATE_K
)
repro_test_cands, repro_test_scores = predict_topk_with_scores(
    bge2sas_repro, test_prompts, item_to_reduced, reduced_to_item, SEQ_LEN, CANDIDATE_K
)
repro_inf_time = time.perf_counter() - t0

repro_val_pred = topk_from_candidates_scores(repro_val_cands, repro_val_scores, TOP_K)
repro_test_pred = topk_from_candidates_scores(repro_test_cands, repro_test_scores, TOP_K)

row_bge2_repro_val = evaluate_predictions(
    model_name="BGE2SASRec-v3-repro-val",
    preds=repro_val_pred,
    gts=val_gts,
    item_counts=item_counts,
    num_items=num_items,
    emb_norm=bge_embeddings,
    item_to_idx=item_to_idx,
    expected_pop_top10=expected_pop_top10,
    alpha=np.nan,
    tau=np.nan,
    training_time=0.0,
    inference_time=0.0,
)
row_bge2_repro_test = evaluate_predictions(
    model_name="BGE2SASRec-v3-repro",
    preds=repro_test_pred,
    gts=test_gts,
    item_counts=item_counts,
    num_items=num_items,
    emb_norm=bge_embeddings,
    item_to_idx=item_to_idx,
    expected_pop_top10=expected_pop_top10,
    alpha=np.nan,
    tau=np.nan,
    training_time=repro_train_time,
    inference_time=repro_inf_time,
)
row_bge2_repro_test = with_v3_meta(
    row_bge2_repro_test,
    embedding_model="bge-m3",
    backbone="SASRec",
    lr_schedule="none",
    early_stop_epoch=np.nan,
)
results_rows.append(row_bge2_repro_test)

repro_fail_reasons = []
if float(row_bge2_repro_test["NDCG@10"]) < V1_REPRO_MIN["NDCG@10"]:
    repro_fail_reasons.append("NDCG@10 below v1 reproduction floor")
if float(row_bge2_repro_test["HR@10"]) < V1_REPRO_MIN["HR@10"]:
    repro_fail_reasons.append("HR@10 below v1 reproduction floor")

print_step_status(
    1,
    "BGE2SASRec-v3-repro",
    row_bge2_repro_test,
    core_targets,
    passed=(len(repro_fail_reasons) == 0),
    fail_reason="; ".join(repro_fail_reasons),
)

ablation_records.append(
    {
        "step": 1,
        "model": "BGE2SASRec-v3-repro",
        "components": "split/regression check",
        "NDCG@10": float(row_bge2_repro_test["NDCG@10"]),
        "HR@10": float(row_bge2_repro_test["HR@10"]),
    }
)


# Step 2: Improved backbone training (scheduler + early stop)
MAX_EPOCHS_V3 = 80 if RUN_MODE == "full" else 50

bge2sas_v3 = build_sasrec_like_model(
    num_items=num_items,
    seq_len=SEQ_LEN,
    emb_dim=EMB_DIM,
    pretrained_item_embeddings=bge_reduced,
)

v3_train_time, v3_best_epoch, v3_lr_schedule = train_with_callbacks(
    model=bge2sas_v3,
    x=x_train,
    y=y_train,
    max_epochs=MAX_EPOCHS_V3,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    early_stop_patience=12 if RUN_MODE != "full" else 18,
    lr_patience=4,
)

t0 = time.perf_counter()
v3_val_cands, v3_val_scores = predict_topk_with_scores(
    bge2sas_v3, val_prompts, item_to_reduced, reduced_to_item, SEQ_LEN, CANDIDATE_K
)
v3_test_cands, v3_test_scores = predict_topk_with_scores(
    bge2sas_v3, test_prompts, item_to_reduced, reduced_to_item, SEQ_LEN, CANDIDATE_K
)
v3_inf_time = time.perf_counter() - t0

v3_val_pred = topk_from_candidates_scores(v3_val_cands, v3_val_scores, TOP_K)
v3_test_pred = topk_from_candidates_scores(v3_test_cands, v3_test_scores, TOP_K)

row_bge2_v3_val = evaluate_predictions(
    model_name="BGE2SASRec-v3-trained-val",
    preds=v3_val_pred,
    gts=val_gts,
    item_counts=item_counts,
    num_items=num_items,
    emb_norm=bge_embeddings,
    item_to_idx=item_to_idx,
    expected_pop_top10=expected_pop_top10,
    alpha=np.nan,
    tau=np.nan,
    training_time=0.0,
    inference_time=0.0,
)
row_bge2_v3_test = evaluate_predictions(
    model_name="BGE2SASRec-v3-trained",
    preds=v3_test_pred,
    gts=test_gts,
    item_counts=item_counts,
    num_items=num_items,
    emb_norm=bge_embeddings,
    item_to_idx=item_to_idx,
    expected_pop_top10=expected_pop_top10,
    alpha=np.nan,
    tau=np.nan,
    training_time=v3_train_time,
    inference_time=v3_inf_time,
)
row_bge2_v3_test = with_v3_meta(
    row_bge2_v3_test,
    embedding_model="bge-m3",
    backbone="SASRec",
    lr_schedule=v3_lr_schedule,
    early_stop_epoch=v3_best_epoch,
)
results_rows.append(row_bge2_v3_test)

step2_ok = float(row_bge2_v3_val["NDCG@10"]) >= 0.060
step2_reason = "validation NDCG@10 < 0.060" if not step2_ok else ""
print_step_status(2, "BGE2SASRec-v3-trained", row_bge2_v3_test, core_targets, step2_ok, step2_reason)

ablation_records.append(
    {
        "step": 2,
        "model": "BGE2SASRec-v3-trained",
        "components": "lr schedule + early stop",
        "NDCG@10": float(row_bge2_v3_test["NDCG@10"]),
        "HR@10": float(row_bge2_v3_test["HR@10"]),
    }
)


# Optional SASRec backbone reference with the same trainer (no pretrained embeddings)
sas_v3 = build_sasrec_like_model(
    num_items=num_items,
    seq_len=SEQ_LEN,
    emb_dim=EMB_DIM,
    pretrained_item_embeddings=None,
)

sas_train_time, sas_best_epoch, sas_lr_schedule = train_with_callbacks(
    model=sas_v3,
    x=x_train,
    y=y_train,
    max_epochs=MAX_EPOCHS_V3,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    early_stop_patience=12 if RUN_MODE != "full" else 18,
    lr_patience=4,
)

t0 = time.perf_counter()
sas_preds_test = predict_topk(sas_v3, test_prompts, item_to_reduced, reduced_to_item, SEQ_LEN, TOP_K)
sas_inf_time = time.perf_counter() - t0

row_sas_v3 = evaluate_predictions(
    model_name="SASRec-v3-trained",
    preds=sas_preds_test,
    gts=test_gts,
    item_counts=item_counts,
    num_items=num_items,
    emb_norm=bge_embeddings,
    item_to_idx=item_to_idx,
    expected_pop_top10=expected_pop_top10,
    alpha=np.nan,
    tau=np.nan,
    training_time=sas_train_time,
    inference_time=sas_inf_time,
)
row_sas_v3 = with_v3_meta(
    row_sas_v3,
    embedding_model="none",
    backbone="SASRec",
    lr_schedule=sas_lr_schedule,
    early_stop_epoch=sas_best_epoch,
)
results_rows.append(row_sas_v3)


# Step 3: Embedding selection (single-pass in fast mode, sweep in full mode)
ENABLE_EMBEDDING_SWEEP = RUN_MODE == "full"
embedding_candidates = ["bge-m3"]
if ENABLE_EMBEDDING_SWEEP:
    embedding_candidates.extend(["gte-large", "e5-large-v2", "nomic-embed-text-v1.5"])

best_payload: dict[str, Any] = {
    "embedding_model": "bge-m3",
    "emb_norm": bge_embeddings,
    "model": bge2sas_v3,
    "train_time": v3_train_time,
    "lr_schedule": v3_lr_schedule,
    "early_stop_epoch": v3_best_epoch,
    "val_row": row_bge2_v3_val,
    "test_row": row_bge2_v3_test,
}

for emb_key in embedding_candidates:
    if emb_key == "bge-m3":
        continue

    print(f"[EMBED] evaluating {emb_key}")
    emb_norm = get_item_embeddings(emb_key, all_item_ids, item_metadata, EMB_DIR)
    emb_reduced = normalize_rows(PCA(n_components=EMB_DIM, random_state=SEED).fit_transform(emb_norm).astype(np.float32))

    emb_model = build_sasrec_like_model(
        num_items=num_items,
        seq_len=SEQ_LEN,
        emb_dim=EMB_DIM,
        pretrained_item_embeddings=emb_reduced,
    )

    emb_train_time, emb_best_epoch, emb_lr_schedule = train_with_callbacks(
        model=emb_model,
        x=x_train,
        y=y_train,
        max_epochs=MAX_EPOCHS_V3,
        batch_size=BATCH_SIZE,
        validation_split=0.1,
        early_stop_patience=12 if RUN_MODE != "full" else 18,
        lr_patience=4,
    )

    t0 = time.perf_counter()
    emb_val_cands, emb_val_scores = predict_topk_with_scores(
        emb_model, val_prompts, item_to_reduced, reduced_to_item, SEQ_LEN, CANDIDATE_K
    )
    emb_test_cands, emb_test_scores = predict_topk_with_scores(
        emb_model, test_prompts, item_to_reduced, reduced_to_item, SEQ_LEN, CANDIDATE_K
    )
    emb_inf_time = time.perf_counter() - t0

    emb_val_pred = topk_from_candidates_scores(emb_val_cands, emb_val_scores, TOP_K)
    emb_test_pred = topk_from_candidates_scores(emb_test_cands, emb_test_scores, TOP_K)

    row_emb_val = evaluate_predictions(
        model_name=f"BGE2SASRec-v3-{emb_key}-val",
        preds=emb_val_pred,
        gts=val_gts,
        item_counts=item_counts,
        num_items=num_items,
        emb_norm=emb_norm,
        item_to_idx=item_to_idx,
        expected_pop_top10=expected_pop_top10,
        alpha=np.nan,
        tau=np.nan,
        training_time=0.0,
        inference_time=0.0,
    )
    row_emb_test = evaluate_predictions(
        model_name=f"BGE2SASRec-v3-{emb_key}",
        preds=emb_test_pred,
        gts=test_gts,
        item_counts=item_counts,
        num_items=num_items,
        emb_norm=emb_norm,
        item_to_idx=item_to_idx,
        expected_pop_top10=expected_pop_top10,
        alpha=np.nan,
        tau=np.nan,
        training_time=emb_train_time,
        inference_time=emb_inf_time,
    )
    row_emb_test = with_v3_meta(
        row_emb_test,
        embedding_model=emb_key,
        backbone="SASRec",
        lr_schedule=emb_lr_schedule,
        early_stop_epoch=emb_best_epoch,
    )
    results_rows.append(row_emb_test)

    cur_val_ndcg = float(row_emb_val["NDCG@10"])
    best_val_ndcg = float(best_payload["val_row"]["NDCG@10"])
    if cur_val_ndcg > best_val_ndcg:
        best_payload = {
            "embedding_model": emb_key,
            "emb_norm": emb_norm,
            "model": emb_model,
            "train_time": emb_train_time,
            "lr_schedule": emb_lr_schedule,
            "early_stop_epoch": emb_best_epoch,
            "val_row": row_emb_val,
            "test_row": row_emb_test,
        }

row_step3 = dict(best_payload["test_row"])
row_step3["model_name"] = "BGE2SASRec-v3-best-embedding"
results_rows.append(row_step3)

step3_ok = float(row_step3["NDCG@10"]) >= 0.058
step3_reason = "best embedding NDCG@10 < 0.058" if not step3_ok else ""
print_step_status(3, "BGE2SASRec-v3-best-embedding", row_step3, core_targets, step3_ok, step3_reason)

ablation_records.append(
    {
        "step": 3,
        "model": "BGE2SASRec-v3-best-embedding",
        "components": f"embedding={best_payload['embedding_model']}",
        "NDCG@10": float(row_step3["NDCG@10"]),
        "HR@10": float(row_step3["HR@10"]),
    }
)

best_model = best_payload["model"]
best_emb_norm = best_payload["emb_norm"]
best_embedding_model = str(best_payload["embedding_model"])

# Build warm candidates once from the selected backbone

# Warm branch remains raw model scores (no score-level reranking)
t0 = time.perf_counter()
base_val_candidates, base_val_scores = predict_topk_with_scores(
    best_model, val_prompts, item_to_reduced, reduced_to_item, SEQ_LEN, CANDIDATE_K
)
base_test_candidates, base_test_scores = predict_topk_with_scores(
    best_model, test_prompts, item_to_reduced, reduced_to_item, SEQ_LEN, CANDIDATE_K
)
base_inf_time = time.perf_counter() - t0


# Step 4: stronger hard ColdBridge search
tau_values = [5, 8, 10, 12, 15, 20]
cold_decay_values = [0.7, 0.8, 0.9]
cold_alpha_values = [0.0, 0.05, 0.1]

cold_val_grid: list[dict[str, Any]] = []
best_cold_cfg: dict[str, Any] | None = None
best_cold_val_row: dict[str, Any] | None = None

for tau, cold_decay, cold_alpha in itertools.product(tau_values, cold_decay_values, cold_alpha_values):
    val_pred = build_full_system_predictions(
        prompts=val_prompts,
        warm_candidates=base_val_candidates,
        warm_scores=base_val_scores,
        all_item_ids=all_item_ids,
        item_to_idx=item_to_idx,
        emb_norm=best_emb_norm,
        item_count_array=item_count_array,
        item_counts=item_counts,
        tau=tau,
        top_k=TOP_K,
        decay=cold_decay,
        alpha_debias=cold_alpha,
    )
    val_row = evaluate_predictions(
        model_name="ColdBridge-v3-val",
        preds=val_pred,
        gts=val_gts,
        item_counts=item_counts,
        num_items=num_items,
        emb_norm=best_emb_norm,
        item_to_idx=item_to_idx,
        expected_pop_top10=expected_pop_top10,
        alpha=cold_alpha,
        tau=tau,
        training_time=0.0,
        inference_time=0.0,
    )

    rec = dict(val_row)
    rec.update({"tau": tau, "cold_decay": cold_decay, "alpha": cold_alpha})
    cold_val_grid.append(rec)

    if best_cold_val_row is None:
        best_cold_val_row = val_row
        best_cold_cfg = {"tau": tau, "cold_decay": cold_decay, "alpha": cold_alpha}
    else:
        cur_tuple = (
            float(val_row["NDCG@10"]),
            float(val_row["HR@10"]),
            float(val_row["CatalogCoverage"]),
        )
        best_tuple = (
            float(best_cold_val_row["NDCG@10"]),
            float(best_cold_val_row["HR@10"]),
            float(best_cold_val_row["CatalogCoverage"]),
        )
        if cur_tuple > best_tuple:
            best_cold_val_row = val_row
            best_cold_cfg = {"tau": tau, "cold_decay": cold_decay, "alpha": cold_alpha}

if best_cold_cfg is None:
    raise RuntimeError("Failed to select ColdBridge-v3 configuration")

cold_test_pred = build_full_system_predictions(
    prompts=test_prompts,
    warm_candidates=base_test_candidates,
    warm_scores=base_test_scores,
    all_item_ids=all_item_ids,
    item_to_idx=item_to_idx,
    emb_norm=best_emb_norm,
    item_count_array=item_count_array,
    item_counts=item_counts,
    tau=int(best_cold_cfg["tau"]),
    top_k=TOP_K,
    decay=float(best_cold_cfg["cold_decay"]),
    alpha_debias=float(best_cold_cfg["alpha"]),
)

row_cold_v3 = evaluate_predictions(
    model_name="ColdBridge-v3",
    preds=cold_test_pred,
    gts=test_gts,
    item_counts=item_counts,
    num_items=num_items,
    emb_norm=best_emb_norm,
    item_to_idx=item_to_idx,
    expected_pop_top10=expected_pop_top10,
    alpha=float(best_cold_cfg["alpha"]),
    tau=int(best_cold_cfg["tau"]),
    training_time=float(best_payload["train_time"]),
    inference_time=base_inf_time,
)
row_cold_v3 = with_v3_meta(
    row_cold_v3,
    embedding_model=best_embedding_model,
    backbone="SASRec",
    lr_schedule=str(best_payload["lr_schedule"]),
    early_stop_epoch=float(best_payload["early_stop_epoch"]),
    blend_type="hard",
    cold_decay=float(best_cold_cfg["cold_decay"]),
)
results_rows.append(row_cold_v3)

step4_ok = (float(row_cold_v3["NDCG@10"]) >= 0.065) and (float(row_cold_v3["HR@10"]) >= 0.110)
step4_reason = "targets: NDCG@10>=0.065 and HR@10>=0.110"
print_step_status(4, "ColdBridge-v3", row_cold_v3, core_targets, step4_ok, "" if step4_ok else step4_reason)

ablation_records.append(
    {
        "step": 4,
        "model": "ColdBridge-v3",
        "components": f"hard tau={best_cold_cfg['tau']}, decay={best_cold_cfg['cold_decay']}, alpha={best_cold_cfg['alpha']}",
        "NDCG@10": float(row_cold_v3["NDCG@10"]),
        "HR@10": float(row_cold_v3["HR@10"]),
    }
)


# Step 5: slot-fill debias on warm branch lower positions
protected_top_grid = [3, 5]
diversity_slots_grid = [3, 5, 7]
tail_threshold_grid = [100, 200, 500]

slot_candidates: list[dict[str, Any]] = []
for protected_top, diversity_slots, tail_threshold in itertools.product(
    protected_top_grid, diversity_slots_grid, tail_threshold_grid
):
    slot_val_pred = build_full_system_predictions(
        prompts=val_prompts,
        warm_candidates=base_val_candidates,
        warm_scores=base_val_scores,
        all_item_ids=all_item_ids,
        item_to_idx=item_to_idx,
        emb_norm=best_emb_norm,
        item_count_array=item_count_array,
        item_counts=item_counts,
        tau=int(best_cold_cfg["tau"]),
        top_k=TOP_K,
        decay=float(best_cold_cfg["cold_decay"]),
        alpha_debias=float(best_cold_cfg["alpha"]),
        protected_top=protected_top,
        diversity_slots=diversity_slots,
        tail_threshold=tail_threshold,
    )

    slot_val_row = evaluate_predictions(
        model_name="ColdBridge-v3-slotfill-val",
        preds=slot_val_pred,
        gts=val_gts,
        item_counts=item_counts,
        num_items=num_items,
        emb_norm=best_emb_norm,
        item_to_idx=item_to_idx,
        expected_pop_top10=expected_pop_top10,
        alpha=float(best_cold_cfg["alpha"]),
        tau=int(best_cold_cfg["tau"]),
        training_time=0.0,
        inference_time=0.0,
    )

    # Keep only candidates that do not hurt validation NDCG vs pure ColdBridge.
    if float(slot_val_row["NDCG@10"]) + 1e-9 < float(best_cold_val_row["NDCG@10"]):
        continue

    score = 0.6 * float(slot_val_row["NDCG@10"]) + 0.4 * float(slot_val_row["LongTail_HR@10"])
    slot_candidates.append(
        {
            "score": score,
            "val_row": slot_val_row,
            "protected_top": protected_top,
            "diversity_slots": diversity_slots,
            "tail_threshold": tail_threshold,
        }
    )

if slot_candidates:
    slot_best = sorted(slot_candidates, key=lambda r: r["score"], reverse=True)[0]
    best_slot_cfg = {
        "protected_top": int(slot_best["protected_top"]),
        "diversity_slots": int(slot_best["diversity_slots"]),
        "tail_threshold": int(slot_best["tail_threshold"]),
    }
    row_slot_val = dict(slot_best["val_row"])
else:
    best_slot_cfg = {"protected_top": 5, "diversity_slots": 5, "tail_threshold": 200}
    row_slot_val = dict(best_cold_val_row)

slot_test_pred = build_full_system_predictions(
    prompts=test_prompts,
    warm_candidates=base_test_candidates,
    warm_scores=base_test_scores,
    all_item_ids=all_item_ids,
    item_to_idx=item_to_idx,
    emb_norm=best_emb_norm,
    item_count_array=item_count_array,
    item_counts=item_counts,
    tau=int(best_cold_cfg["tau"]),
    top_k=TOP_K,
    decay=float(best_cold_cfg["cold_decay"]),
    alpha_debias=float(best_cold_cfg["alpha"]),
    protected_top=int(best_slot_cfg["protected_top"]),
    diversity_slots=int(best_slot_cfg["diversity_slots"]),
    tail_threshold=int(best_slot_cfg["tail_threshold"]),
)

row_slot_v3 = evaluate_predictions(
    model_name="ColdBridge-v3-slotfill",
    preds=slot_test_pred,
    gts=test_gts,
    item_counts=item_counts,
    num_items=num_items,
    emb_norm=best_emb_norm,
    item_to_idx=item_to_idx,
    expected_pop_top10=expected_pop_top10,
    alpha=float(best_cold_cfg["alpha"]),
    tau=int(best_cold_cfg["tau"]),
    training_time=float(best_payload["train_time"]),
    inference_time=base_inf_time,
)
row_slot_v3 = with_v3_meta(
    row_slot_v3,
    embedding_model=best_embedding_model,
    backbone="SASRec",
    lr_schedule=str(best_payload["lr_schedule"]),
    early_stop_epoch=float(best_payload["early_stop_epoch"]),
    blend_type="hard+slot_fill",
    debias_variant="slot_fill",
    cold_decay=float(best_cold_cfg["cold_decay"]),
    protected_top=int(best_slot_cfg["protected_top"]),
    diversity_slots=int(best_slot_cfg["diversity_slots"]),
    tail_threshold=int(best_slot_cfg["tail_threshold"]),
)
results_rows.append(row_slot_v3)

step5_ok = float(row_slot_v3["NDCG@10"]) + 1e-9 >= float(row_cold_v3["NDCG@10"])
step5_reason = "slot-fill reduced test NDCG@10" if not step5_ok else ""
print_step_status(5, "ColdBridge-v3-slotfill", row_slot_v3, core_targets, step5_ok, step5_reason)

ablation_records.append(
    {
        "step": 5,
        "model": "ColdBridge-v3-slotfill",
        "components": (
            f"slot-fill protected_top={best_slot_cfg['protected_top']}, "
            f"diversity_slots={best_slot_cfg['diversity_slots']}, "
            f"tail_threshold={best_slot_cfg['tail_threshold']}"
        ),
        "NDCG@10": float(row_slot_v3["NDCG@10"]),
        "HR@10": float(row_slot_v3["HR@10"]),
    }
)


# Step 6: hybrid position ensemble with LLMSeqSim-style candidates
seq_positions_grid = [10, 12, 15, 17]
hybrid_candidates: list[dict[str, Any]] = []

for seq_positions in seq_positions_grid:
    hybrid_val_pred = build_full_system_predictions(
        prompts=val_prompts,
        warm_candidates=base_val_candidates,
        warm_scores=base_val_scores,
        all_item_ids=all_item_ids,
        item_to_idx=item_to_idx,
        emb_norm=best_emb_norm,
        item_count_array=item_count_array,
        item_counts=item_counts,
        tau=int(best_cold_cfg["tau"]),
        top_k=TOP_K,
        decay=float(best_cold_cfg["cold_decay"]),
        alpha_debias=float(best_cold_cfg["alpha"]),
        protected_top=int(best_slot_cfg["protected_top"]),
        diversity_slots=int(best_slot_cfg["diversity_slots"]),
        tail_threshold=int(best_slot_cfg["tail_threshold"]),
        seq_positions=int(seq_positions),
    )

    hybrid_val_row = evaluate_predictions(
        model_name="ColdBridge-v3-hybrid-val",
        preds=hybrid_val_pred,
        gts=val_gts,
        item_counts=item_counts,
        num_items=num_items,
        emb_norm=best_emb_norm,
        item_to_idx=item_to_idx,
        expected_pop_top10=expected_pop_top10,
        alpha=float(best_cold_cfg["alpha"]),
        tau=int(best_cold_cfg["tau"]),
        training_time=0.0,
        inference_time=0.0,
    )

    hybrid_candidates.append({"seq_positions": seq_positions, "val_row": hybrid_val_row})

hybrid_best = sorted(
    hybrid_candidates,
    key=lambda x: (
        float(x["val_row"]["NDCG@10"]),
        float(x["val_row"]["HR@10"]),
    ),
    reverse=True,
)[0]

best_seq_positions = int(hybrid_best["seq_positions"])
row_hybrid_val = dict(hybrid_best["val_row"])

hybrid_test_pred = build_full_system_predictions(
    prompts=test_prompts,
    warm_candidates=base_test_candidates,
    warm_scores=base_test_scores,
    all_item_ids=all_item_ids,
    item_to_idx=item_to_idx,
    emb_norm=best_emb_norm,
    item_count_array=item_count_array,
    item_counts=item_counts,
    tau=int(best_cold_cfg["tau"]),
    top_k=TOP_K,
    decay=float(best_cold_cfg["cold_decay"]),
    alpha_debias=float(best_cold_cfg["alpha"]),
    protected_top=int(best_slot_cfg["protected_top"]),
    diversity_slots=int(best_slot_cfg["diversity_slots"]),
    tail_threshold=int(best_slot_cfg["tail_threshold"]),
    seq_positions=best_seq_positions,
)

row_hybrid_v3 = evaluate_predictions(
    model_name="ColdBridge-v3-hybrid",
    preds=hybrid_test_pred,
    gts=test_gts,
    item_counts=item_counts,
    num_items=num_items,
    emb_norm=best_emb_norm,
    item_to_idx=item_to_idx,
    expected_pop_top10=expected_pop_top10,
    alpha=float(best_cold_cfg["alpha"]),
    tau=int(best_cold_cfg["tau"]),
    training_time=float(best_payload["train_time"]),
    inference_time=base_inf_time,
)
row_hybrid_v3 = with_v3_meta(
    row_hybrid_v3,
    embedding_model=best_embedding_model,
    backbone="SASRec",
    lr_schedule=str(best_payload["lr_schedule"]),
    early_stop_epoch=float(best_payload["early_stop_epoch"]),
    blend_type="hard+slot_fill+hybrid",
    debias_variant="slot_fill",
    cold_decay=float(best_cold_cfg["cold_decay"]),
    protected_top=int(best_slot_cfg["protected_top"]),
    diversity_slots=int(best_slot_cfg["diversity_slots"]),
    tail_threshold=int(best_slot_cfg["tail_threshold"]),
    seq_positions=best_seq_positions,
)
results_rows.append(row_hybrid_v3)

step6_ok = float(row_hybrid_v3["NDCG@10"]) + 1e-9 >= float(row_slot_v3["NDCG@10"])
step6_reason = "hybrid reduced test NDCG@10" if not step6_ok else ""
print_step_status(6, "ColdBridge-v3-hybrid", row_hybrid_v3, core_targets, step6_ok, step6_reason)

ablation_records.append(
    {
        "step": 6,
        "model": "ColdBridge-v3-hybrid",
        "components": f"hybrid seq_positions={best_seq_positions}",
        "NDCG@10": float(row_hybrid_v3["NDCG@10"]),
        "HR@10": float(row_hybrid_v3["HR@10"]),
    }
)


# Step 7: FULL_SYSTEM_v3 chosen by validation NDCG among step-4/5/6 variants
final_candidates = [
    ("ColdBridge-v3", best_cold_val_row, row_cold_v3),
    ("ColdBridge-v3-slotfill", row_slot_val, row_slot_v3),
    ("ColdBridge-v3-hybrid", row_hybrid_val, row_hybrid_v3),
]

best_name, best_final_val, best_final_test = sorted(
    final_candidates,
    key=lambda x: (
        float(x[1]["NDCG@10"]),
        float(x[1]["HR@10"]),
    ),
    reverse=True,
)[0]

row_full_v3 = dict(best_final_test)
row_full_v3["model_name"] = "FULL_SYSTEM_v3"
results_rows.append(row_full_v3)

step7_ok = float(row_full_v3["NDCG@10"]) >= 0.065
step7_reason = "FULL_SYSTEM_v3 NDCG@10 < 0.065" if not step7_ok else ""
print_step_status(7, "FULL_SYSTEM_v3", row_full_v3, core_targets, step7_ok, step7_reason)

ablation_records.append(
    {
        "step": 7,
        "model": "FULL_SYSTEM_v3",
        "components": f"selected={best_name}",
        "NDCG@10": float(row_full_v3["NDCG@10"]),
        "HR@10": float(row_full_v3["HR@10"]),
    }
)


# Save ablation v3 markdown
ablation_path = RESULTS_DIR / "ablation_v3.md"
ablation_lines = [
    "# Ablation v3",
    "",
    "| Step | Model | Components | NDCG@10 | HR@10 |",
    "|---|---|---|---:|---:|",
]
for rec in sorted(ablation_records, key=lambda r: int(r["step"])):
    ablation_lines.append(
        f"| {rec['step']} | {rec['model']} | {rec['components']} | {float(rec['NDCG@10']):.6f} | {float(rec['HR@10']):.6f} |"
    )
ablation_path.write_text("\n".join(ablation_lines) + "\n", encoding="utf-8")


# Pareto fronts (v3 and compatibility v2 filename)
pareto_input_rows = [
    row_pop,
    row_sas_v3,
    row_bge2_repro_test,
    row_bge2_v3_test,
    row_step3,
    row_cold_v3,
    row_slot_v3,
    row_hybrid_v3,
    row_full_v3,
]
pareto_input_df = pd.DataFrame(pareto_input_rows)

if len(pareto_input_df) > 0:
    pareto_mask = find_pareto_front(pareto_input_df)
    pareto_df = pareto_input_df.loc[pareto_mask].copy()
else:
    pareto_df = pd.DataFrame()

pareto_v3_csv = RESULTS_DIR / "pareto_front_v3.csv"
pareto_v2_csv = RESULTS_DIR / "pareto_front_v2.csv"
pareto_df.to_csv(pareto_v3_csv, index=False)
pareto_df.to_csv(pareto_v2_csv, index=False)


# Append v3 results to full_results.csv with extended schema
results_csv = RESULTS_DIR / "full_results.csv"
schema_cols = make_extended_schema_cols()

new_results_df = pd.DataFrame(results_rows)
for c in schema_cols:
    if c not in new_results_df.columns:
        new_results_df[c] = np.nan
new_results_df = new_results_df[schema_cols]

if results_csv.exists():
    old_df = pd.read_csv(results_csv)
    for c in schema_cols:
        if c not in old_df.columns:
            old_df[c] = np.nan
    old_df = old_df[schema_cols]
    merged_df = pd.concat([old_df, new_results_df], ignore_index=True)
else:
    merged_df = new_results_df.copy()

merged_df.to_csv(results_csv, index=False)


# Write best_model.txt in v3 format
best_txt = RESULTS_DIR / "best_model.txt"
llm2 = core_targets.get("LLM2SASRec", CORE_TARGETS_DEFAULT["LLM2SASRec"])
bert4 = core_targets.get("BERT4Rec", CORE_TARGETS_DEFAULT["BERT4Rec"])
core_sas = core_targets.get("SASRec", CORE_TARGETS_DEFAULT["SASRec"])

best_ndcg = float(row_full_v3["NDCG@10"])
best_hr = float(row_full_v3["HR@10"])
best_cov = float(row_full_v3["CatalogCoverage"])
best_lt = float(row_full_v3["LongTail_HR@10"])
best_ild = float(row_full_v3["ILD@10"])

debias_label = "slot-fill" if str(row_full_v3.get("debias_variant", "")).startswith("slot") else "none"
ensemble_label = "hybrid" if np.isfinite(_safe_float(row_full_v3.get("seq_positions", np.nan))) else "none"

lines = [
    "=== BEST MODEL v3 ===",
    f"Model:      {row_full_v3.get('model_name', 'FULL_SYSTEM_v3')}",
    f"Backbone:   {row_full_v3.get('backbone', 'SASRec')}",
    f"Embedding:  {row_full_v3.get('embedding_model', 'bge-m3')}",
    f"ColdBridge: tau={row_full_v3.get('tau', np.nan)}",
    f"Debias:     {debias_label}",
    f"Ensemble:   {ensemble_label}",
    "",
    f"NDCG@10:        {best_ndcg:.6f}",
    f"  vs Core LLM2SASRec:  {pct_delta(best_ndcg, float(llm2['NDCG@10'])):+.2f}%  BEATS: {'YES' if best_ndcg > float(llm2['NDCG@10']) else 'NO'}",
    f"  vs Core SASRec:      {pct_delta(best_ndcg, float(core_sas['NDCG@10'])):+.2f}%  BEATS: {'YES' if best_ndcg > float(core_sas['NDCG@10']) else 'NO'}",
    f"HR@10:          {best_hr:.6f}",
    f"  vs Core LLM2SASRec:  {pct_delta(best_hr, float(llm2['HR@10'])):+.2f}%  BEATS: {'YES' if best_hr > float(llm2['HR@10']) else 'NO'}",
    f"CatalogCoverage:{best_cov:.6f}",
    f"LongTail_HR@10: {best_lt:.6f}",
    f"ILD@10:         {best_ild:.6f}",
    "",
    f"Reference Core BERT4Rec NDCG@10: {float(bert4['NDCG@10']):.5f}",
]
best_txt.write_text("\n".join(lines) + "\n", encoding="utf-8")


# Figures: tau sweep and ablation progression
if cold_val_grid:
    tau_df = pd.DataFrame(cold_val_grid)
    tau_best = (
        tau_df.sort_values(["tau", "NDCG@10", "HR@10"], ascending=[True, False, False])
        .groupby("tau", as_index=False)
        .first()
    )

    plt.figure(figsize=(7, 5))
    plt.plot(tau_best["tau"], tau_best["NDCG@10"], marker="o", label="NDCG@10")
    plt.plot(tau_best["tau"], tau_best["HR@10"], marker="s", label="HR@10")
    plt.xlabel("tau")
    plt.ylabel("metric")
    plt.title("ColdBridge-v3 tau sweep")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / "tau_sweep_v3.png", dpi=180)
    plt.close()

abl_df = pd.DataFrame(ablation_records).sort_values("step")
if not abl_df.empty:
    plt.figure(figsize=(8, 5))
    plt.plot(abl_df["step"], abl_df["NDCG@10"], marker="o", label="NDCG@10")
    plt.plot(abl_df["step"], abl_df["HR@10"], marker="s", label="HR@10")
    plt.xticks(abl_df["step"].tolist())
    plt.xlabel("Ablation step")
    plt.ylabel("Metric")
    plt.title("Ablation v3 progression")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / "ablation_v3_progress.png", dpi=180)
    plt.close()


print("Done.")
print("Results CSV:", results_csv)
print("Best model:", best_txt)
print("Ablation v3:", ablation_path)
print("Pareto v3:", pareto_v3_csv)
print("Pareto v2 (compat):", pareto_v2_csv)
print("Figures dir:", FIG_DIR)

display(merged_df.sort_values(["NDCG@10", "HR@10"], ascending=False).head(40))

Train sessions: 1598
Val sessions: 228
Test sessions: 457


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Train examples: (69582, 20) (69582,)


I0000 00:00:1776690186.020090      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0
I0000 00:00:1776690189.691606     761 service.cc:152] XLA service 0x78eb840036e0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776690189.691652     761 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1776690190.255541     761 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1776690193.458428     761 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Best alpha: 0.1
Best tau: 15
Best variant: log_norm alpha: 0.1
Best decay: 1.0
Done.
Results CSV: /kaggle/working/llmseqrec_ml100k_single_notebook/results/movielens_100k/full_results.csv
Best model: /kaggle/working/llmseqrec_ml100k_single_notebook/results/movielens_100k/best_model.txt
Figures dir: /kaggle/working/llmseqrec_ml100k_single_notebook/results/figures


,model_name,alpha,tau,NDCG@10,NDCG@20,HR@10,HR@20,LongTail_HR@10,CatalogCoverage,Serendipity,ILD@10,training_time_sec,inference_time_sec,debias_variant,cold_decay
11,ColdBridge-BGE2SASRec,NaN,15.0,0.058122,0.071388,0.094092,0.148796,0.094092,0.528886,0.091904,0.173683,14.476991,1.698569,NaN,NaN
9,ColdBridge-BGE2SASRec,NaN,5.0,0.057809,0.070646,0.094092,0.146608,0.094092,0.515783,0.091904,0.187545,14.476991,1.698569,NaN,NaN
10,ColdBridge-BGE2SASRec,NaN,10.0,0.057325,0.070592,0.091904,0.146608,0.091904,0.527099,0.089716,0.178454,14.476991,1.698569,NaN,NaN
8,ColdBridge-BGE2SASRec,NaN,3.0,0.054375,0.065616,0.091904,0.137856,0.091904,0.506849,0.087527,0.192811,14.476991,1.698569,NaN,NaN
7,ColdBridge-BGE2SASRec,NaN,2.0,0.052916,0.064265,0.091904,0.137856,0.091904,0.492555,0.087527,0.198264,14.476991,1.698569,NaN,NaN
1,SASRec,NaN,NaN,0.047220,0.055436,0.100656,0.133479,0.100656,0.301370,0.089716,0.213208,15.279092,0.939186,NaN,NaN
2,BGE2SASRec,NaN,NaN,0.046117,0.058570,0.085339,0.135667,0.085339,0.317451,0.076586,0.213129,14.476991,1.698569,NaN,NaN
16,PopDebias-ColdBridge-rank,0.1,15.0,0.031374,0.043817,0.052516,0.102845,0.052516,0.372245,0.039387,0.174766,14.476991,1.698569,rank,NaN
17,PopDebias-ColdBridge-rank,0.3,15.0,0.031129,0.042968,0.050328,0.098468,0.050328,0.379988,0.037199,0.178092,14.476991,1.698569,rank,NaN
18,PopDebias-ColdBridge-rank,0.5,15.0,0.031129,0.043381,0.050328,0.100656,0.050328,0.381179,0.037199,0.178709,14.476991,1.698569,rank,NaN


In [ ]:
# Optional: quick artifact preview after the main run cell
results_csv = RESULTS_DIR / "full_results.csv"
best_txt = RESULTS_DIR / "best_model.txt"
figures_dir = FIG_DIR
ablation_md = RESULTS_DIR / "ablation_v3.md"
pareto_v3_csv = RESULTS_DIR / "pareto_front_v3.csv"
pareto_v2_csv = RESULTS_DIR / "pareto_front_v2.csv"

if results_csv.exists():
    df = pd.read_csv(results_csv)
    display(df.sort_values(["NDCG@10", "HR@10"], ascending=False).head(30))
else:
    print("Results file not found:", results_csv)

if best_txt.exists():
    print("=== best_model.txt ===")
    print(best_txt.read_text(encoding="utf-8"))
else:
    print("Best-model file not found:", best_txt)

if ablation_md.exists():
    print("=== ablation_v3.md ===")
    print(ablation_md.read_text(encoding="utf-8"))
else:
    print("Ablation file not found:", ablation_md)

if pareto_v3_csv.exists():
    print("=== pareto_front_v3.csv (head) ===")
    display(pd.read_csv(pareto_v3_csv).head(20))
else:
    print("Pareto v3 file not found:", pareto_v3_csv)

if pareto_v2_csv.exists():
    print("=== pareto_front_v2.csv (head) ===")
    display(pd.read_csv(pareto_v2_csv).head(20))
else:
    print("Pareto v2 file not found:", pareto_v2_csv)

if figures_dir.exists():
    print("Figures:")
    for p in sorted(figures_dir.glob("*.png")):
        print("-", p.name)

,model_name,alpha,tau,NDCG@10,NDCG@20,HR@10,HR@20,LongTail_HR@10,CatalogCoverage,Serendipity,ILD@10,training_time_sec,inference_time_sec,debias_variant,cold_decay
0,ColdBridge-BGE2SASRec,NaN,15.0,0.058122,0.071388,0.094092,0.148796,0.094092,0.528886,0.091904,0.173683,14.476991,1.698569,NaN,NaN
1,ColdBridge-BGE2SASRec,NaN,5.0,0.057809,0.070646,0.094092,0.146608,0.094092,0.515783,0.091904,0.187545,14.476991,1.698569,NaN,NaN
2,ColdBridge-BGE2SASRec,NaN,10.0,0.057325,0.070592,0.091904,0.146608,0.091904,0.527099,0.089716,0.178454,14.476991,1.698569,NaN,NaN
3,ColdBridge-BGE2SASRec,NaN,3.0,0.054375,0.065616,0.091904,0.137856,0.091904,0.506849,0.087527,0.192811,14.476991,1.698569,NaN,NaN
4,ColdBridge-BGE2SASRec,NaN,2.0,0.052916,0.064265,0.091904,0.137856,0.091904,0.492555,0.087527,0.198264,14.476991,1.698569,NaN,NaN
5,SASRec,NaN,NaN,0.047220,0.055436,0.100656,0.133479,0.100656,0.301370,0.089716,0.213208,15.279092,0.939186,NaN,NaN
6,BGE2SASRec,NaN,NaN,0.046117,0.058570,0.085339,0.135667,0.085339,0.317451,0.076586,0.213129,14.476991,1.698569,NaN,NaN
7,PopDebias-ColdBridge-rank,0.1,15.0,0.031374,0.043817,0.052516,0.102845,0.052516,0.372245,0.039387,0.174766,14.476991,1.698569,rank,NaN
8,PopDebias-ColdBridge-rank,0.3,15.0,0.031129,0.042968,0.050328,0.098468,0.050328,0.379988,0.037199,0.178092,14.476991,1.698569,rank,NaN
9,PopDebias-ColdBridge-rank,0.5,15.0,0.031129,0.043381,0.050328,0.100656,0.050328,0.381179,0.037199,0.178709,14.476991,1.698569,rank,NaN


=== best_model.txt ===
BEST MODEL: ColdBridge-BGE2SASRec
Best Alpha: nan
Best Tau:   15.0
NDCG@10:    0.058122 (vs baseline: 0.046117, delta: +26.03%)
HR@10:      0.094092
LT-HR@10:   0.094092
ILD@10:     0.173683

Figures:
- alpha_sweep_movielens_100k.png
- cold_warm_gap_movielens_100k.png
- tau_sweep_movielens_100k.png


v3 is implemented in the main run cell with these stages:

1. Step-0/1 regression diagnostics and reproduction check for BGE2SASRec.
2. Improved backbone training with learning-rate scheduling and early stopping.
3. Optional embedding sweep (enabled automatically when `RUN_MODE = "full"`).
4. Hard ColdBridge-v3 search (tau, cold decay, mild cold-branch debias).
5. Slot-fill debiasing on warm-branch lower positions only.
6. Hybrid position ensemble with LLMSeqSim-style candidates.
7. FULL_SYSTEM_v3 selection and artifact export.

Key outputs:
- `results/movielens_100k/full_results.csv`
- `results/movielens_100k/best_model.txt`
- `results/movielens_100k/ablation_v3.md`
- `results/movielens_100k/pareto_front_v3.csv`
- `results/movielens_100k/pareto_front_v2.csv` (compatibility alias)

Runtime switches on Kaggle:
- keep default fast mode: `NOVELTY_RUN_MODE=v3_fast`
- full embedding sweep: `NOVELTY_RUN_MODE=full`
- fail early when Step-1 reproduction breaks: `ENFORCE_REPRO_GATE=1`